# Carga incremental completa · v1.0

Un único notebook con las dos partes del proceso, en el orden correcto:

**Parte 1 (Celdas 1-6)** — carga documental: clientes, proveedores,
contratos, facturación línea a línea, y reconciliación de las tablas
puente (a qué inmueble y qué tipo de operación corresponde cada
artículo).

**Parte 2 (Celdas 7-11)** — reconciliación de cobros y pagos: para
cada factura, cuánto se ha cobrado/pagado de verdad y cuánto queda
pendiente, siguiendo fielmente los mecanismos de compensación reales
de FacturaPlus (documentados con datos, no supuestos).

**Parte 3 (Celda 12)** — cuadre final: todo lo que ha pasado por
cualquier combinación `cref`+`cprop` (todas las líneas de `Facclil.dbf`
y `Facprol.dbf`) queda o imputado a un inmueble, o sin inmueble con un
motivo registrado, o descartado por el cargador con su motivo. La suma
cuadra al céntimo y se contrasta con la base de datos.

**Ejecutar siempre con "Reiniciar y ejecutar todas las celdas"**,
nunca celdas sueltas — así se garantiza que cada paso ve los datos ya
guardados por el paso anterior, sin nada cacheado en memoria de una
ejecución previa.


## Celda 1 — Conexión y funciones base

Lo mínimo que necesita cualquier otra celda para funcionar: la
conexión a la base de datos, cómo leer un `.dbf`, y cómo imprimir el
resumen de cada carga. Las funciones que solo hacen falta en una
celda concreta (limpiar nombres, `CNUMFACREC`...) están definidas
justo donde se usan, no aquí.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
from dbfread import DBF
from sqlalchemy import create_engine, text

# La contraseña NO se escribe aquí: se lee de la variable de entorno PGPASSWORD.
PG_USER, PG_HOST, PG_PORT, PG_DB = os.environ.get("PGUSER", "postgres"), "localhost", "5432", "bi_inmobiliario"
PG_PASS = os.environ.get("PGPASSWORD", "")
engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}")

# Aquí dejan las funciones de facturación lo que han resuelto en esta ejecución
# (las líneas de origen, lo descartado y cada línea con su inmueble), para que
# la última celda pueda cuadrar el total. No se guarda en ninguna tabla.
CUADRE = {}

# Freno de seguridad: si faltan en el origen MÁS facturas que estas respecto a lo ya guardado, la
# carga se detiene con un error en lugar de borrar. Un DBF leído vacío o a medias haría que
# "todo lo que falta" pareciera borrado. Si de verdad has borrado muchas facturas en FacturaPlus
# (por ejemplo, al depurar años antiguos), sube este número y ejecuta de nuevo.
MAX_FACTURAS_A_BORRAR = 50

# Lo mismo para los contratos: si faltan en el origen MÁS contratos que estos respecto a lo ya guardado,
# la carga se detiene sin borrar nada (un Atipicas.dbf leído vacío o a medias haría que "todo" pareciera borrado).
MAX_CONTRATOS_A_BORRAR = 20

RUTA_FACTURAPLUS = r"C:\ruta\a\los\dbf\de\facturaplus"
RUTA_PROCESADA = r"C:\ruta\a\la\carpeta\de\los\excel"

# Excel con el catálogo de inmuebles. Basta el nombre si está en RUTA_PROCESADA; si está en otra carpeta, pon aquí la ruta completa.
ARCHIVO_INMUEBLES = "inmuebles.xlsx"

# Qué hacer cuando un campo de un inmueble está VACÍO en el Excel pero ya tiene valor en la base de datos:
#   True  = se conserva el valor de la base y se avisa (protege contra una celda borrada por error)
#   False = el vacío del Excel manda y el valor se borra de la base
PROTEGER_CAMPOS_VACIOS = True


def leer_dbf(nombre):
    # Abre un .dbf y lo deja listo para trabajar: quita espacios en
    # blanco sobrantes de las columnas de texto (FacturaPlus los deja
    # con relleno fijo), y convierte CCONTADOR a número entero — en
    # algunos ficheros llega como texto, y sin esta conversión los
    # cruces con otras tablas por esta columna fallarían en silencio.
    tabla = DBF(f"{RUTA_FACTURAPLUS}\\{nombre}", encoding='cp850')
    df = pd.DataFrame(iter(tabla))
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    if 'CCONTADOR' in df.columns:
        df['CCONTADOR'] = df['CCONTADOR'].astype(int)
    return df


def informe(tabla, nuevos=0, actualizados=0, avisos=0, eliminados=0):
    # La línea de resumen que se ve al final de cada carga, con
    # formato fijo para que todas las tablas queden alineadas:
    #   clientes                     | nuevos:      7 | actualizados:      0 | avisos:    0
    # "nuevos" son filas insertadas por primera vez; "actualizados"
    # son filas que ya existían y cuyo valor ha cambiado; "avisos" es
    # el número de casos que necesitan revisión manual (por ejemplo,
    # una combinación cref+cprop que aún no existe en ninguna tabla
    # puente); "eliminados" son filas borradas de verdad (solo lo
    # usan las funciones que reemplazan documentos completos borrando
    # primero -- el resto de funciones, que solo hacen INSERT/UPDATE,
    # lo dejan en 0 y ni siquiera se imprime la columna). No lanza
    # ningún error ni detiene nada -- es solo el informe visual de
    # qué ha pasado en esta ejecución.
    linea = f"{tabla:<28} | nuevos: {nuevos:>6,} | actualizados: {actualizados:>6,} | avisos: {avisos:>4,}"
    if eliminados > 0:
        linea += f" | eliminados: {eliminados:>6,}"
    print(linea)


def a_centimos(serie):
    # Importes a céntimos ENTEROS. Sumar enteros es exacto; sumar decimales
    # en coma flotante deja residuos (del tipo 2,27e-13) que impedirían
    # comprobar que un total cuadra "al céntimo".
    return np.rint(pd.to_numeric(serie, errors='coerce').fillna(0).astype(float) * 100).astype('int64')


## Celda 2 — ARTICULOS_INMUEBLES y ARTICULOS_TIPO_OPERACION

Las dos tablas "puente" que traducen un artículo de FacturaPlus
(`cref`+`cprop`) a un inmueble concreto, y a una categoría de
operación (alquiler, gastos de comunidad...). Se cargan desde dos
Excel de referencia mantenidos a mano.


In [ ]:
TIPOS_VALIDOS = {'ALQUILER', 'GASTOS_EXPLOTACION', 'GASTOS_MANTENIMIENTO', 'IMPUESTOS_TASAS',
                 'COMPRAVENTA', 'GASTOS_PROMOCION', 'GASTOS_ESTRUCTURA', 'OTROS'}


def revisar_puente(df, tabla, claves, tipos_validos=None):
    # Solo AVISA (no cambia nada en la base de datos). Detecta lo que el cargador
    # no puede ver por sí solo: claves repetidas en el Excel, espacios sobrantes
    # (que impiden cruzar con nada), tipos de operación desconocidos y, sobre
    # todo, filas que siguen en la base de datos pero ya NO están en el Excel
    # (el cargador nunca borra, y una fila huérfana puede seguir imputando
    # líneas a un inmueble sin que nadie lo vea). Devuelve cuántos avisos hay.
    avisos = 0
    dup = df[df.duplicated(claves, keep=False)]
    if len(dup) > 0:
        avisos += 1
        print(f"⚠️ AVISO [{tabla}]: {len(dup)} filas del Excel repiten la clave {claves} (se queda con la última):")
        print(dup.to_string(index=False))
    for col in df.columns:
        txt = df[col].astype(str)
        con_espacios = df[(txt != txt.str.strip()) & df[col].notna()]
        if len(con_espacios) > 0:
            avisos += 1
            print(f"⚠️ AVISO [{tabla}]: {len(con_espacios)} valores de '{col}' con espacios al principio o al final (no cruzarán con nada):")
            print(con_espacios.head(15).to_string(index=False))
    if tipos_validos is not None:
        raros = df[~df['tipo_operacion'].isin(tipos_validos)]
        if len(raros) > 0:
            avisos += 1
            print(f"⚠️ AVISO [{tabla}]: {len(raros)} filas con un tipo de operación que no es ninguno de los conocidos:")
            print(raros.to_string(index=False))
    bd = pd.read_sql(f"SELECT {', '.join(claves)} FROM {tabla}", engine)
    en_excel = set(map(tuple, df[claves].fillna('').astype(str).values))
    huerfanas = bd[[tuple(r) not in en_excel for r in bd[claves].fillna('').astype(str).values]]
    if len(huerfanas) > 0:
        avisos += 1
        print(f"⚠️ AVISO [{tabla}]: {len(huerfanas)} filas están en la base de datos pero YA NO en el Excel "
              f"(el cargador no borra: si siguen activas, pueden seguir imputando líneas):")
        print(huerfanas.head(30).to_string(index=False))
        if len(huerfanas) > 30:
            print(f"   ... y {len(huerfanas) - 30} más")
    return avisos


COLUMNAS_INMUEBLES = ['id_inmueble', 'promocion', 'tipo', 'descripcion', 'planta', 'puerta',
                      'superficie_construida', 'num_dormitorios', 'num_banos',
                      'terraza_patio_interior', 'superficie_terraza_patio_interior',
                      'finca_registral', 'referencia_catastral', 'direccion', 'municipio',
                      'codigo_postal', 'estado_especial', 'coste_adquisicion', 'fecha_adquisicion', 'notas']


def cargar_inmuebles():
    # Lee el Excel de inmuebles y deja la tabla inmuebles al día: inserta los que no existen y actualiza los que
    # han cambiado. Va la PRIMERA de todas las cargas porque articulos_inmuebles, contratos y facturación apuntan
    # a estos identificadores (clave foránea). Nunca borra: un inmueble que ya no está en el Excel solo genera un aviso.
    # Basta el nombre si el Excel está en RUTA_PROCESADA; si está en otra carpeta, ARCHIVO_INMUEBLES puede ser la ruta completa
    ruta = ARCHIVO_INMUEBLES if os.path.isabs(ARCHIVO_INMUEBLES) else f"{RUTA_PROCESADA}\\{ARCHIVO_INMUEBLES}"
    df = pd.read_excel(ruta)
    faltan = [c for c in COLUMNAS_INMUEBLES if c not in df.columns]
    if faltan:
        raise ValueError(f"Al Excel de inmuebles ({ARCHIVO_INMUEBLES}) le faltan estas columnas: {faltan}")
    df = df[COLUMNAS_INMUEBLES].copy()
    avisos = 0

    # finca_registral es un identificador, no una cantidad: un entero disfrazado de decimal (79180.0) pierde el .0;
    # si ya tiene otro formato (59348/23), se deja tal cual
    def limpiar_finca(x):
        if pd.isna(x):
            return None
        t = str(x).strip()
        return t[:-2] if t.endswith('.0') and t[:-2].isdigit() else t
    df['finca_registral'] = df['finca_registral'].apply(limpiar_finca)

    # Identificador: se recortan espacios, se ignoran las filas sin id y, si hay ids repetidos, manda la última fila
    df['id_inmueble'] = df['id_inmueble'].apply(lambda x: None if pd.isna(x) else str(x))
    limpio = df['id_inmueble'].str.strip()
    con_espacios = df[df['id_inmueble'].notna() & (df['id_inmueble'] != limpio)]
    if len(con_espacios) > 0:
        avisos += 1
        print(f"⚠️ AVISO [inmuebles]: {len(con_espacios)} identificadores con espacios al principio o al final (se han recortado): "
              f"{list(con_espacios['id_inmueble'])[:15]}")
    df['id_inmueble'] = limpio
    sin_id = df[df['id_inmueble'].isna() | (df['id_inmueble'] == '')]
    if len(sin_id) > 0:
        avisos += 1
        print(f"⚠️ AVISO [inmuebles]: {len(sin_id)} filas del Excel sin id_inmueble (se ignoran):")
        print(sin_id[['promocion', 'tipo', 'puerta']].head(15).to_string())
        df = df.drop(sin_id.index)
    dup = df[df.duplicated('id_inmueble', keep=False)]
    if len(dup) > 0:
        avisos += 1
        print(f"⚠️ AVISO [inmuebles]: {len(dup)} filas del Excel repiten el id_inmueble (se queda con la última):")
        print(dup[['id_inmueble', 'promocion', 'tipo', 'puerta']].to_string(index=False))
        df = df.drop_duplicates('id_inmueble', keep='last')

    columnas = COLUMNAS_INMUEBLES
    sin_pk = columnas[1:]
    antes = pd.read_sql(f"SELECT {', '.join(columnas)} FROM inmuebles", engine).set_index('id_inmueble')
    registros = df.astype(object).where(df.notna(), None).to_dict('records')

    # Campos vacíos en el Excel que ya tienen valor en la base: con la protección activada, se conserva el de la base
    if PROTEGER_CAMPOS_VACIOS:
        protegidos = [{'id_inmueble': r['id_inmueble'], 'columna': c, 'valor_conservado': antes.at[r['id_inmueble'], c]}
                      for r in registros if r['id_inmueble'] in antes.index
                      for c in sin_pk if r[c] is None and pd.notna(antes.at[r['id_inmueble'], c])]
        if protegidos:
            avisos += 1
            print(f"⚠️ AVISO [inmuebles]: {len(protegidos)} campos vacíos en el Excel se han PROTEGIDO (se conserva el valor de la base de datos). "
                  f"Si el vacío es intencionado, pon PROTEGER_CAMPOS_VACIOS = False en la Celda 1:")
            print(pd.DataFrame(protegidos).head(30).to_string(index=False))
            if len(protegidos) > 30:
                print(f"   ... y {len(protegidos) - 30} más")

    # El valor que queda tras la carga: con la protección, el de la base si el Excel viene vacío
    valor = (lambda c: f"COALESCE(EXCLUDED.{c}, inmuebles.{c})") if PROTEGER_CAMPOS_VACIOS else (lambda c: f"EXCLUDED.{c}")
    consulta = text(f"""
        INSERT INTO inmuebles ({', '.join(columnas)})
        VALUES ({', '.join(':' + c for c in columnas)})
        ON CONFLICT (id_inmueble) DO UPDATE SET {', '.join(f'{c} = {valor(c)}' for c in sin_pk)}, fecha_actualiz = now()
        WHERE ({', '.join(f'inmuebles.{c}' for c in sin_pk)}) IS DISTINCT FROM ({', '.join(valor(c) for c in sin_pk)})
        RETURNING id_inmueble, (xmax = 0) AS es_nuevo
    """)
    nuevos, actualizados = [], []
    with engine.begin() as conn:           # todo o nada: si falla una fila, no se escribe ninguna
        for r in registros:
            fila = conn.execute(consulta, r).fetchone()
            if fila is not None:
                (nuevos if fila[1] else actualizados).append(fila[0])

    # Qué ha cambiado exactamente (se compara la base de datos con la base de datos, así no hay diferencias de formato)
    if nuevos or actualizados:
        despues = pd.read_sql(f"SELECT {', '.join(columnas)} FROM inmuebles", engine).set_index('id_inmueble')
        def ver(v):
            if pd.isna(v):
                return 'vacío'
            return str(int(v)) if isinstance(v, float) and v.is_integer() else str(v)
        if nuevos:
            print(f"   Nuevos: " + ', '.join(f"{i} ({despues.at[i, 'promocion']}, {despues.at[i, 'tipo']})" for i in nuevos[:30]) + (f" ... y {len(nuevos) - 30} más" if len(nuevos) > 30 else ''))
        lineas = []
        for i in actualizados:
            for c in sin_pk:
                a, b = antes.at[i, c], despues.at[i, c]
                if (pd.isna(a) != pd.isna(b)) or (pd.notna(a) and a != b):
                    lineas.append(f"   Actualizado {i} · {c}: {ver(a)} → {ver(b)}")
        for l in lineas[:30]:
            print(l)
        if len(lineas) > 30:
            print(f"   ... y {len(lineas) - 30} cambios más")

    # El cargador nunca borra: un inmueble de la base que ya no está en el Excel solo se avisa
    huerfanos = antes[~antes.index.isin(df['id_inmueble'])]
    if len(huerfanos) > 0:
        avisos += 1
        print(f"⚠️ AVISO [inmuebles]: {len(huerfanos)} inmuebles están en la base de datos pero YA NO en el Excel (no se borran):")
        print(huerfanos[['promocion', 'tipo', 'puerta']].head(15).to_string())
        if len(huerfanos) > 15:
            print(f"   ... y {len(huerfanos) - 15} más")
    informe('inmuebles', len(nuevos), len(actualizados), avisos)


def cargar_articulos_inmuebles():
    # dtype=str fuerza a que cref/cprop se lean como texto, aunque
    # "parezcan" números (p.ej. "0013") -- si no, pandas podría leerlos
    # como enteros y perder el cero a la izquierda, rompiendo cualquier
    # cruce posterior con el resto de tablas
    df = pd.read_excel(f"{RUTA_PROCESADA}\\articulos_inmuebles.xlsx", dtype={'cref': str, 'cprop': str})
    # Un cprop en blanco se representa siempre como '' (nunca NULL) --
    # en SQL, comparar cualquier cosa contra NULL nunca da "verdadero",
    # así que dejarlo en blanco de verdad rompería el respaldo genérico
    df['cprop'] = df['cprop'].fillna('')
    # Los huecos en id_inmueble pueden llegar como NaN (numpy) en vez
    # de None (Python) tras leer el Excel -- NaN no se traduce bien a
    # NULL de PostgreSQL, así que se normaliza explícitamente a None
    df['id_inmueble'] = pd.Series([None if pd.isna(v) else v for v in df['id_inmueble']], index=df.index, dtype=object)
    # Fila de encabezado/nota suelta que a veces queda en el Excel, sin
    # ser un artículo real
    df = df[df['cref'] != 'Rectificación'].copy()

    # Todo id_inmueble del Excel debe existir ya en la tabla inmuebles (clave foránea). Se comprueba ANTES de
    # escribir nada para dar un mensaje claro en lugar de un error técnico de la base de datos.
    existentes = set(pd.read_sql("SELECT id_inmueble FROM inmuebles", engine)['id_inmueble'])
    sin_inmueble = df[df['id_inmueble'].notna() & ~df['id_inmueble'].isin(existentes)]
    if len(sin_inmueble) > 0:
        raise ValueError(f"articulos_inmuebles.xlsx usa {sin_inmueble['id_inmueble'].nunique()} id_inmueble que NO existen en la tabla inmuebles: "
                         f"{sorted(sin_inmueble['id_inmueble'].unique())[:20]}. Añádelos al Excel de inmuebles ({ARCHIVO_INMUEBLES}) "
                         f"y ejecuta de nuevo. Filas afectadas (cref, cprop): {list(zip(sin_inmueble['cref'], sin_inmueble['cprop']))[:10]}")

    nuevos, actualizados = 0, 0
    with engine.begin() as conn:
        for _, row in df.iterrows():
            # ON CONFLICT DO UPDATE con un WHERE que compara el valor
            # nuevo contra el actual: si es idéntico, Postgres NO
            # actualiza la fila ni la devuelve en RETURNING — así se
            # sabe distinguir "no había nada que hacer" de "se acaba
            # de tocar la fila", sin necesitar una consulta aparte.
            # xmax=0 es la forma de preguntarle a Postgres "¿esta fila
            # se acaba de INSERTAR en esta misma sentencia, o ya
            # existía y se ha hecho UPDATE?".
            r = conn.execute(text("""
                INSERT INTO articulos_inmuebles (cref, cprop, id_inmueble)
                VALUES (:cref, :cprop, :id_inmueble)
                ON CONFLICT (cref, cprop) DO UPDATE SET
                    id_inmueble = EXCLUDED.id_inmueble, fecha_actualiz = now()
                WHERE articulos_inmuebles.id_inmueble IS DISTINCT FROM EXCLUDED.id_inmueble
                RETURNING (xmax = 0) AS es_nuevo
            """), {k: (None if pd.isna(v) else v) for k, v in row.to_dict().items()})
            fila = r.fetchone()
            if fila is not None:
                nuevos += 1 if fila[0] else 0
                actualizados += 0 if fila[0] else 1

    informe("articulos_inmuebles", nuevos, actualizados, revisar_puente(df, "articulos_inmuebles", ["cref", "cprop"]))


def cargar_articulos_tipo_operacion():
    # Mismo patrón que la función anterior, para la tabla puente
    # cref -> tipo_operacion (esta no necesita cprop, porque el tipo
    # de operación depende solo del artículo, no de la unidad concreta).
    df = pd.read_excel(f"{RUTA_PROCESADA}\\articulos_tipo_operacion.xlsx", dtype={'cref': str})

    nuevos, actualizados = 0, 0
    with engine.begin() as conn:
        for _, row in df.iterrows():
            r = conn.execute(text("""
                INSERT INTO articulos_tipo_operacion (cref, tipo_operacion)
                VALUES (:cref, :tipo_operacion)
                ON CONFLICT (cref) DO UPDATE SET
                    tipo_operacion = EXCLUDED.tipo_operacion, fecha_actualiz = now()
                WHERE articulos_tipo_operacion.tipo_operacion IS DISTINCT FROM EXCLUDED.tipo_operacion
                RETURNING (xmax = 0) AS es_nuevo
            """), {k: (None if pd.isna(v) else v) for k, v in row.to_dict().items()})
            fila = r.fetchone()
            if fila is not None:
                nuevos += 1 if fila[0] else 0
                actualizados += 0 if fila[0] else 1

    informe("articulos_tipo_operacion", nuevos, actualizados, revisar_puente(df, "articulos_tipo_operacion", ["cref"], TIPOS_VALIDOS))


## Celda 3 — CLIENTES y PROVEEDORES

Carga directa desde `Clientes.dbf`/`Proveedo.dbf`, con el mismo
patrón `INSERT`/`UPDATE` condicional de la celda anterior. Aquí viven
las funciones de limpieza de texto que solo se usan al cargar
nombres: `limpiar_nombre` (corrige tildes/eñes mal codificadas) y
`clasificar_documento` (DNI/NIE/empresa, a partir del formato del NIF).


In [ ]:
# Tabla de correcciones: cada símbolo roto conocido -> la letra que
# debería ser. Se completa a mano cada vez que aparece un caso nuevo.
DICCIONARIO_LIMPIEZA = {
    '┌': 'Ú', '═': 'Í', '╓': 'Í', '╔': 'É', 'Ë': 'Ó', '╙': 'Ó',
    '▄': 'Ü', 'Ð': 'Ñ', '┴': 'Á', '└': 'À', 'M¬': 'Mª', 'M═': 'Mª', '║': 'º',
}


def limpiar_nombre(serie_nombres):
    # Los nombres llegan del DBF con tildes/eñes mal codificadas
    # (símbolos sueltos en vez de la letra real) — este diccionario
    # traduce cada símbolo roto conocido a su letra correcta.
    s = serie_nombres.astype(str).str.strip().str.upper().str.lstrip("'").str.strip()
    for malo, bueno in DICCIONARIO_LIMPIEZA.items():
        s = s.str.replace(malo, bueno, regex=False)
    return s


PERMITIDOS_NO_ASCII = set('ÁÉÍÓÚÀÈÒÜÑÇªº°·')


def avisar_simbolos_raros(nombres, tabla):
    # Solo AVISA. Tras la limpieza, un nombre que aún lleve un símbolo fuera del
    # ASCII y de las letras españolas habituales es una tilde mal leída que el
    # diccionario todavía no conoce: sin este aviso pasaría a la base de datos en silencio.
    raros = nombres[nombres.apply(lambda s: any(ord(c) > 127 and c not in PERMITIDOS_NO_ASCII for c in s))]
    if len(raros) > 0:
        print(f"⚠️ AVISO [{tabla}]: {len(raros)} nombres con símbolos no reconocidos (si son tildes mal leídas, añádelas a DICCIONARIO_LIMPIEZA):")
        print(raros.head(15).to_string())
    return len(raros)


def clasificar_documento(nif):
    # Distingue DNI/NIE/empresa solo a partir del FORMATO del NIF
    # (8 dígitos+letra, letra+7 dígitos+letra, o letra+7 dígitos+dígito
    # o letra de control) — sin validar el dígito de control en sí.
    import re
    if pd.isna(nif) or str(nif).strip() == '':
        return 'DESCONOCIDO'
    nif = str(nif).strip().upper()
    if re.match(r'^[0-9]{8}[A-Z]$', nif):
        return 'DNI'
    if re.match(r'^[XYZ][0-9]{7}[A-Z]$', nif):
        return 'NIE'
    if re.match(r'^[ABCDEFGHJKLMNPQRSUVW][0-9]{7}[0-9A-J]$', nif):
        return 'EMPRESA'
    return 'DESCONOCIDO'


def avisar_terceros_huerfanos(codigos_origen, tabla, columna):
    # Solo AVISA (no borra). Un cliente/proveedor que sigue en la base de datos pero ya no está en el DBF
    # (borrado, o con el código cambiado en FacturaPlus) no desaparece solo: se queda sin actividad, y si
    # tiene contratos o facturas antiguas conviene revisarlo. Devuelve 1 si hay aviso, 0 si no.
    bd = pd.read_sql(f"SELECT {columna} FROM {tabla}", engine)[columna]
    huerfanos = bd[~bd.isin(set(codigos_origen))]
    if len(huerfanos) == 0:
        return 0
    print(f"⚠️ AVISO [{tabla}]: {len(huerfanos)} códigos están en la base de datos pero YA NO en el DBF (no se borran): "
          f"{sorted(huerfanos)[:15]}" + (f" ... y {len(huerfanos) - 15} más" if len(huerfanos) > 15 else ''))
    return 1


def cargar_clientes():
    df_raw = leer_dbf("Clientes.dbf")
    # Quedarse solo con las columnas que interesan, y renombrarlas ya
    # con el nombre real de la tabla en PostgreSQL
    df = df_raw[['CCODCLI', 'CNOMCLI', 'CTFO1CLI', 'CTFO2CLI', 'CDNICIF', 'EMAIL']].copy()
    df.columns = ['ccodcli', 'nombre_completo', 'telefono1', 'telefono2', 'nif', 'email']
    # Limpiar TODOS los nombres de golpe (vectorizado, mucho más
    # rápido que limpiar cliente a cliente), y clasificar el tipo de
    # documento de cada uno a partir de su NIF -- todo esto ANTES de
    # abrir ninguna conexión con la base de datos
    df['nombre_completo'] = limpiar_nombre(df['nombre_completo'])
    df['tipo_documento'] = df['nif'].apply(clasificar_documento)
    simbolos_raros = avisar_simbolos_raros(df['nombre_completo'], 'clientes')

    nuevos, actualizados = 0, 0
    with engine.begin() as conn:
        for _, row in df.iterrows():
            # Mismo patrón INSERT/ON CONFLICT que en la Celda 2, pero
            # comparando varias columnas a la vez (fila completa entre
            # paréntesis) en vez de una sola.
            r = conn.execute(text("""
                INSERT INTO clientes (ccodcli, nombre_completo, telefono1, telefono2, nif, email, tipo_documento)
                VALUES (:ccodcli, :nombre_completo, :telefono1, :telefono2, :nif, :email, :tipo_documento)
                ON CONFLICT (ccodcli) DO UPDATE SET
                    nombre_completo = EXCLUDED.nombre_completo, telefono1 = EXCLUDED.telefono1,
                    telefono2 = EXCLUDED.telefono2, nif = EXCLUDED.nif, email = EXCLUDED.email,
                    tipo_documento = EXCLUDED.tipo_documento, fecha_actualiz = now()
                WHERE (clientes.nombre_completo, clientes.telefono1, clientes.telefono2,
                       clientes.nif, clientes.email, clientes.tipo_documento)
                      IS DISTINCT FROM
                      (EXCLUDED.nombre_completo, EXCLUDED.telefono1, EXCLUDED.telefono2,
                       EXCLUDED.nif, EXCLUDED.email, EXCLUDED.tipo_documento)
                RETURNING (xmax = 0) AS es_nuevo
            """), row.to_dict())
            fila = r.fetchone()
            if fila is not None:
                nuevos += 1 if fila[0] else 0
                actualizados += 0 if fila[0] else 1

    informe("clientes", nuevos, actualizados, simbolos_raros + avisar_terceros_huerfanos(df['ccodcli'], 'clientes', 'ccodcli'))


def cargar_proveedores():
    # Mismo patrón que cargar_clientes(), sin tipo_documento (no se
    # usa para proveedores).
    df_raw = leer_dbf("Proveedo.dbf")
    df = df_raw[['CCODPRO', 'CNOMPRO', 'CTFO1PRO', 'CTFO2PRO', 'CNIFDNI', 'EMAIL']].copy()
    df.columns = ['ccodpro', 'nombre_completo', 'telefono1', 'telefono2', 'nif', 'email']
    # Limpiar todos los nombres de golpe, igual que en cargar_clientes()
    df['nombre_completo'] = limpiar_nombre(df['nombre_completo'])
    simbolos_raros = avisar_simbolos_raros(df['nombre_completo'], 'proveedores')

    nuevos, actualizados = 0, 0
    with engine.begin() as conn:
        for _, row in df.iterrows():
            r = conn.execute(text("""
                INSERT INTO proveedores (ccodpro, nombre_completo, telefono1, telefono2, nif, email)
                VALUES (:ccodpro, :nombre_completo, :telefono1, :telefono2, :nif, :email)
                ON CONFLICT (ccodpro) DO UPDATE SET
                    nombre_completo = EXCLUDED.nombre_completo, telefono1 = EXCLUDED.telefono1,
                    telefono2 = EXCLUDED.telefono2, nif = EXCLUDED.nif, email = EXCLUDED.email,
                    fecha_actualiz = now()
                WHERE (proveedores.nombre_completo, proveedores.telefono1, proveedores.telefono2,
                       proveedores.nif, proveedores.email)
                      IS DISTINCT FROM
                      (EXCLUDED.nombre_completo, EXCLUDED.telefono1, EXCLUDED.telefono2,
                       EXCLUDED.nif, EXCLUDED.email)
                RETURNING (xmax = 0) AS es_nuevo
            """), row.to_dict())
            fila = r.fetchone()
            if fila is not None:
                nuevos += 1 if fila[0] else 0
                actualizados += 0 if fila[0] else 1

    informe("proveedores", nuevos, actualizados, simbolos_raros + avisar_terceros_huerfanos(df['ccodpro'], 'proveedores', 'ccodpro'))


## Celda 4 — CONTRATOS (carga + agrupación)

Dos pasos sobre la misma tabla, uno detrás de otro:

**1. `cargar_contratos()`** — cruza `Atipicas.dbf` con `clientes` (por
`CCODCLI`) y con `articulos_inmuebles` (por `CREF`+`CPROP`) para
resolver `id_cliente`/`id_inmueble`. La identidad de un contrato es
`id_cliente + cref + cprop + fecha_inicio` — si ya existe esa
combinación, se actualiza (precio, fecha_fin, o `id_inmueble` si
antes estaba en blanco); si no, se crea uno nuevo.

**2. `agrupar_contratos()`** — asigna un `id_grupo_contrato` a
contratos relacionados (mismo inmueble desde la misma fecha, por
ejemplo dos cotitulares de un mismo alquiler), para poder verlos como
una única unidad de negocio en vez de filas sueltas.

Van en la misma celda porque el segundo paso siempre depende del
resultado del primero (necesita que `id_inmueble` ya esté resuelto).


In [ ]:
def cargar_contratos():
    df = leer_dbf("Atipicas.dbf")

    clientes_bd = pd.read_sql("SELECT id_cliente, ccodcli FROM clientes", engine)
    df['CCODCLI'] = df['CCODCLI'].astype(str).str.zfill(6)
    df = df.merge(clientes_bd, left_on='CCODCLI', right_on='ccodcli', how='left')

    df['CPROP1'] = df['CPROP1'].fillna('')
    articulos_bd = pd.read_sql("SELECT cref, cprop, id_inmueble FROM articulos_inmuebles", engine)
    df = df.merge(
        articulos_bd, left_on=['CREF', 'CPROP1'], right_on=['cref', 'cprop'], how='left', indicator='_en_puente'
    )

    nuevos_df = df[['id_cliente', 'id_inmueble', 'NPRECIO', 'DFECINI', 'DFECFIN', 'CREF', 'CPROP1']].copy()
    nuevos_df.columns = ['id_cliente', 'id_inmueble', 'precio', 'fecha_inicio', 'fecha_fin', 'cref', 'cprop']
    # Redondear ANTES de comparar precios — si no, un contrato con
    # 300.001 en el DBF y 300.00 en la base de datos (la columna
    # NUMERIC ya redondeada) se marcaría como "cambiado" sin serlo.
    nuevos_df['precio'] = nuevos_df['precio'].round(2)

    existentes = pd.read_sql(
        "SELECT id_contrato, id_cliente, id_inmueble, cref, cprop, precio, fecha_inicio, fecha_fin FROM contratos",
        engine
    )

    # La identidad de un contrato: mismo cliente, mismo artículo
    # (cref+cprop), misma fecha de inicio. Si coincide, es el MISMO
    # contrato (aunque cambien precio o fecha_fin); si no coincide
    # ninguna combinación, es un contrato nuevo.
    def clave(d):
        return d['id_cliente'].astype(str) + '|' + d['cref'] + '|' + d['cprop'] + '|' + d['fecha_inicio'].astype(str)

    existentes['clave'] = clave(existentes)
    nuevos_df['clave'] = clave(nuevos_df)
    avisos = 0

    # Dos filas del DBF con la misma identidad no se pueden distinguir entre sí (ni aquí ni en la base de datos).
    duplicadas = nuevos_df[nuevos_df.duplicated('clave', keep=False)]
    if len(duplicadas) > 0:
        avisos += 1
        print(f"⚠️ AVISO [contratos]: {len(duplicadas)} filas de Atipicas.dbf repiten cliente + artículo + fecha de inicio "
              f"(no se pueden distinguir entre sí):")
        print(duplicadas[['id_cliente', 'cref', 'cprop', 'fecha_inicio', 'fecha_fin', 'precio']].to_string(index=False))

    # Contratos de la base de datos que YA NO están en el origen: el cliente, el artículo o la fecha de inicio
    # cambiaron en FacturaPlus (o el contrato se borró). Como esos tres datos forman la identidad del contrato,
    # el cambio se ve como "contrato nuevo" y el antiguo se quedaría en la base contando dos veces. Se eliminan,
    # salvo que falten demasiados de golpe: entonces lo normal es que el DBF se haya leído mal y se detiene todo.
    obsoletos = existentes[~existentes['clave'].isin(nuevos_df['clave'])]
    if len(nuevos_df) == 0 and len(existentes) > 0:
        raise RuntimeError(f"[contratos] Atipicas.dbf no trae ningún contrato y la base de datos tiene {len(existentes)}. "
                           f"No se ha borrado nada: revisa que el DBF de origen esté completo.")
    if len(obsoletos) > MAX_CONTRATOS_A_BORRAR:
        raise RuntimeError(f"[contratos] Faltan en el origen {len(obsoletos)} contratos que SÍ están en la base de datos "
                           f"(límite de seguridad: {MAX_CONTRATOS_A_BORRAR}). No se ha borrado nada. Si es correcto, sube "
                           f"MAX_CONTRATOS_A_BORRAR en la Celda 1; si no, revisa que Atipicas.dbf esté completo. "
                           f"Primeros id_contrato: {obsoletos['id_contrato'].head(5).tolist()}")

    solo_nuevos = nuevos_df[~nuevos_df['clave'].isin(existentes['clave'])].drop(columns='clave')

    # Para los que YA existían: comparar si algo cambió (precio,
    # fecha_fin, o a qué inmueble corresponde el artículo) y
    # actualizar solo esos.
    coincidencias = nuevos_df.merge(
        existentes[['id_contrato', 'clave', 'id_inmueble', 'precio', 'fecha_fin']],
        on='clave', suffixes=('_nuevo', '_actual')
    )
    cambia_precio = (
        pd.to_numeric(coincidencias['precio_actual'], errors='coerce') -
        pd.to_numeric(coincidencias['precio_nuevo'], errors='coerce')
    ).abs() > 0.001
    # Dos fechas vacías NO son un cambio (en pandas, None != None es verdadero, y un contrato
    # vigente, sin fecha de fin, se marcaba como modificado en cada ejecución).
    cambia_fecha_fin = ~((coincidencias['fecha_fin_nuevo'] == coincidencias['fecha_fin_actual'])
                         | (coincidencias['fecha_fin_nuevo'].isna() & coincidencias['fecha_fin_actual'].isna()))
    # El inmueble se actualiza si ahora se puede resolver y es distinto del guardado (antes solo si el guardado
    # estaba en blanco: si en la tabla puente se corregía a qué inmueble corresponde un artículo, el contrato
    # conservaba el antiguo). Si el artículo ya no está en la tabla puente, se conserva el que había.
    inmueble_nuevo = coincidencias['id_inmueble_nuevo']
    cambia_inmueble = inmueble_nuevo.notna() & (inmueble_nuevo != coincidencias['id_inmueble_actual'].fillna('__NULL__'))
    a_actualizar = coincidencias[cambia_precio | cambia_fecha_fin | cambia_inmueble]

    # Todo en UNA sola transacción: o se borran los obsoletos, se insertan los nuevos y se actualizan los
    # cambiados, o no se toca nada.
    actualizados = 0
    with engine.begin() as conn:
        if len(obsoletos) > 0:
            conn.execute(text("DELETE FROM contratos WHERE id_contrato IN :ids").bindparams(
                ids=tuple(int(i) for i in obsoletos['id_contrato'])))
        if len(solo_nuevos) > 0:
            solo_nuevos.to_sql('contratos', conn, if_exists='append', index=False)
        if len(a_actualizar) > 0:
            temp = a_actualizar[['id_contrato', 'id_inmueble_nuevo', 'precio_nuevo', 'fecha_fin_nuevo']].rename(
                columns={'id_inmueble_nuevo': 'id_inmueble', 'precio_nuevo': 'precio', 'fecha_fin_nuevo': 'fecha_fin'}
            )
            temp.to_sql('_temp_contratos', conn, if_exists='replace', index=False)
            r = conn.execute(text("""
                UPDATE contratos c SET
                    id_inmueble = COALESCE(t.id_inmueble, c.id_inmueble),
                    precio = t.precio, fecha_fin = t.fecha_fin::date, fecha_actualiz = now()
                FROM _temp_contratos t WHERE c.id_contrato = t.id_contrato
            """))
            actualizados = r.rowcount
            conn.execute(text("DROP TABLE _temp_contratos"))

    # Qué se ha eliminado y qué inmuebles han cambiado, para poder comprobarlo a ojo
    if len(obsoletos) > 0:
        codigos = clientes_bd.set_index('id_cliente')['ccodcli']
        listado = obsoletos.assign(cliente=obsoletos['id_cliente'].map(codigos))[
            ['id_contrato', 'cliente', 'id_inmueble', 'cref', 'cprop', 'precio', 'fecha_inicio', 'fecha_fin']]
        print(f"   Contratos eliminados porque ya no están en el origen ({len(obsoletos)}):")
        print(listado.to_string(index=False))
    movidos = coincidencias[cambia_inmueble & coincidencias['id_inmueble_actual'].notna()]
    if len(movidos) > 0:
        print(f"   Contratos que cambian de inmueble según la tabla puente ({len(movidos)}):")
        print(movidos[['id_contrato', 'cref', 'cprop', 'id_inmueble_actual', 'id_inmueble_nuevo']].to_string(index=False))

    sin_revisar = df[df['_en_puente'] == 'left_only'][['CREF', 'CPROP1']].drop_duplicates()
    informe("contratos", len(solo_nuevos), actualizados, len(sin_revisar) + avisos, len(obsoletos))
    if len(sin_revisar) > 0:
        print(sin_revisar.to_string(index=False))


def agrupar_contratos():
    df_completo = pd.read_sql("""
        SELECT c.id_contrato, c.id_cliente, c.id_inmueble, c.fecha_inicio, c.fecha_fin,
               i.promocion, i.puerta, i.tipo
        FROM contratos c
        LEFT JOIN inmuebles i ON c.id_inmueble = i.id_inmueble
    """, engine)

    # Sin id_inmueble, no hay nada que agrupar — queda en NULL directamente.
    df = df_completo[df_completo['id_inmueble'].notna()].copy()

    # Union-Find clásico: cada contrato empieza siendo su propio grupo
    # (padre = sí mismo), y "unir" fusiona dos grupos en uno.
    padre = {i: i for i in df['id_contrato']}

    def encontrar(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    def unir(x, y):
        rx, ry = encontrar(x), encontrar(y)
        if rx != ry:
            padre[rx] = ry

    # Paso 1 — agrupar por inmueble+fecha_inicio: todas las filas que
    # alquilan el mismo inmueble desde la misma fecha son un mismo
    # "sub-grupo" (esto ya cubre parejas, incluidas separaciones donde
    # uno de los dos se va antes con una fecha_fin distinta —
    # confirmado con el caso real de un inmueble).
    #
    # No se exige que fecha_fin también coincida: el caso "distinto
    # de verdad" que se detectó (otro inmueble) resultó ser un problema de
    # datos — dos inmuebles reales compartiendo por error el mismo
    # id_inmueble en la tabla puente, no una ambigüedad de fechas — y
    # se corrige en origen, no con lógica de agrupación.
    df['clave_inmueble'] = df['id_inmueble'].astype(str) + '|' + df['fecha_inicio'].astype(str)
    for _, filas_inmueble in df.groupby('clave_inmueble'):
        ids = filas_inmueble['id_contrato'].tolist()
        for i in range(1, len(ids)):
            unir(ids[0], ids[i])

    # Paso 2 — fusionar dos sub-grupos de inmueble SOLO si tienen
    # exactamente el mismo conjunto de clientes (no basta con
    # compartir uno) — evita mezclar, por ejemplo, un trastero que un
    # cliente alquila solo con una vivienda que alquila junto a su
    # pareja el mismo día.
    df['raiz_inmueble'] = df['id_contrato'].apply(encontrar)
    clientes_por_subgrupo = df.groupby('raiz_inmueble')['id_cliente'].apply(lambda s: frozenset(s)).to_dict()
    fecha_por_subgrupo = df.groupby('raiz_inmueble')['fecha_inicio'].first().to_dict()

    subgrupos_por_fecha = {}
    for raiz, fecha in fecha_por_subgrupo.items():
        subgrupos_por_fecha.setdefault(fecha, []).append(raiz)

    for fecha, raices_fecha in subgrupos_por_fecha.items():
        for i in range(len(raices_fecha)):
            for j in range(i + 1, len(raices_fecha)):
                r1, r2 = raices_fecha[i], raices_fecha[j]
                if clientes_por_subgrupo[r1] == clientes_por_subgrupo[r2]:
                    unir(r1, r2)

    raices = sorted(set(encontrar(i) for i in df['id_contrato']))

    # Construir el nombre descriptivo: G<ancla> <Promoción> <puerta1 + puerta2 + ...>
    # El ancla es el id_contrato más bajo del grupo — nunca cambia
    # entre ejecuciones, así que el nombre es estable en el tiempo
    # aunque se recalculen los grupos. Las puertas se ordenan por
    # tipo (Vivienda primero, Garaje después, Trastero al final) para
    # que el nombre se lea de forma natural.
    orden_tipo = {'Vivienda': 0, 'Garaje': 1, 'Trastero': 2}
    mapa_grupo = {}
    for raiz in raices:
        filas_grupo = df[df['id_contrato'].apply(encontrar) == raiz]
        ancla = filas_grupo['id_contrato'].min()
        promocion = filas_grupo['promocion'].dropna().iloc[0] if filas_grupo['promocion'].notna().any() else "SIN_PROMOCION"
        inmuebles_grupo = filas_grupo[['tipo', 'puerta']].dropna(subset=['puerta']).drop_duplicates()
        inmuebles_grupo['_orden'] = inmuebles_grupo['tipo'].map(orden_tipo).fillna(99)
        inmuebles_grupo = inmuebles_grupo.sort_values('_orden')
        descripciones = inmuebles_grupo['puerta'].tolist()
        texto_inmuebles = " + ".join(descripciones) if descripciones else "SIN_INMUEBLE"
        mapa_grupo[raiz] = f"G{ancla} {promocion} {texto_inmuebles}"[:250]  # margen amplio, con varios inmuebles por grupo

    df['id_grupo_contrato'] = df['id_contrato'].apply(lambda i: mapa_grupo[encontrar(i)])

    # Volver a juntar con las filas sin inmueble, que quedan en NULL
    sin_inmueble = df_completo[df_completo['id_inmueble'].isna()][['id_contrato']].copy()
    sin_inmueble['id_grupo_contrato'] = None
    df_final = pd.concat([df[['id_contrato', 'id_grupo_contrato']], sin_inmueble], ignore_index=True)

    temp = df_final.copy()
    temp.to_sql('_temp_grupos', engine, if_exists='replace', index=False)
    with engine.begin() as conn:
        r = conn.execute(text("""
            UPDATE contratos c SET id_grupo_contrato = t.id_grupo_contrato
            FROM _temp_grupos t WHERE c.id_contrato = t.id_contrato
            AND c.id_grupo_contrato IS DISTINCT FROM t.id_grupo_contrato
        """))
        actualizados = r.rowcount
        conn.execute(text("DROP TABLE _temp_grupos"))

    informe("agrupar_contratos", 0, actualizados)


## Celda 5 — FACTURACION_CLIENTES y FACTURACION_PROVEEDORES

Carga el detalle línea a línea de cada factura (`Facclil.dbf` /
`Facprol.dbf`), resolviendo `id_cliente`/`id_proveedor`,
`id_inmueble` (vía `articulos_inmuebles`) y `tipo_operacion`.

**Detecta automáticamente altas, modificaciones y borrados de
líneas** — no solo documentos nuevos. Como el origen no tiene ningún
campo propio que identifique "esta línea concreta" de forma estable
(comprobado: `Facclil.dbf` no tiene ningún `NLINEA`/`NORDEN`), no se
puede actualizar línea a línea con total seguridad — si se borra una
línea del medio, las siguientes se desplazan de posición.

**La solución, 100% automática y sin ambigüedad, y sin necesitar
ninguna columna nueva**: comparar la "huella" de todas las líneas de
la factura juntas — usando el **orden natural de lectura** del DBF
para los datos nuevos, y **`id_linea` ascendente** (ya existente)
para reconstruir el orden original de lo ya guardado. Si la huella no
coincide exactamente, se borra la factura entera y se reinserta tal
cual está en el DBF ahora — nunca hay que adivinar qué línea es cuál.


In [ ]:
def huellas_por_documento(d, col_orden, col_prefijo=None):
    # Para cada factura, una cadena con TODAS sus líneas ("inmueble:tipo:importe:fecha", con el nº de
    # factura del proveedor delante en proveedores), en el orden de col_orden. Formato idéntico al del
    # bucle original (NULL para vacíos, importe con 2 decimales, fecha AAAA-MM-DD), pero calculado
    # por columnas en lugar de línea a línea: es lo que más tardaba en toda la carga documental.
    d = d.sort_values(col_orden)
    inm = d['id_inmueble'].astype(object).where(d['id_inmueble'].notna(), 'NULL').astype(str)
    tipo = d['tipo_operacion'].astype(object).where(d['tipo_operacion'].notna(), 'NULL').astype(str)
    importe = pd.to_numeric(d['importe_linea'], errors='coerce')
    imp = pd.Series(np.char.mod('%.2f', importe.to_numpy(dtype=float)), index=d.index).where(importe.notna(), 'NULL')
    fecha = pd.to_datetime(d['fecha'], errors='coerce')
    fec = fecha.dt.strftime('%Y-%m-%d').where(fecha.notna(), 'NULL')
    parte = inm + ':' + tipo + ':' + imp + ':' + fec
    if col_prefijo:
        parte = d[col_prefijo].astype(str) + ':' + parte
    return parte.groupby(d['documento'], sort=False).agg('||'.join)


def documentos_con_otro_tercero(nuevo_df, existente_df):
    # El cliente/proveedor de la cabecera no forma parte de la huella de líneas. Se compara aparte:
    # así un cambio de cliente en FacturaPlus (o un cliente dado de alta después de cargar la
    # factura) se propaga, sin reescribir todas las facturas por cambiar el formato de la huella.
    col = 'id_cliente' if 'id_cliente' in nuevo_df.columns else 'id_proveedor'
    if len(existente_df) == 0:
        return []
    _id = lambda v: 'NULL' if pd.isna(v) else str(int(v))
    nuevo = nuevo_df.groupby('documento', sort=False)[col].first().map(_id)
    actual = existente_df.groupby('documento', sort=False)[col].first().map(_id)
    comunes = nuevo.index.intersection(actual.index)
    return [d for d in comunes if nuevo[d] != actual[d]]


def controlar_desapariciones(a_reemplazar, existian, nuevo_df):
    # Una factura que está en la base de datos y ya no está en el origen se borra. Si faltan
    # demasiadas de golpe, lo normal es que el DBF se haya leído mal: se detiene todo.
    tabla = 'facturacion_clientes' if 'id_cliente' in nuevo_df.columns else 'facturacion_proveedores'
    en_origen = set(nuevo_df['documento'])
    desaparecen = [d for d in a_reemplazar if d in existian and d not in en_origen]
    if len(desaparecen) > MAX_FACTURAS_A_BORRAR:
        raise RuntimeError(f"[{tabla}] Faltan en el origen {len(desaparecen)} facturas que SÍ están en la base de datos "
                           f"(límite de seguridad: {MAX_FACTURAS_A_BORRAR}). No se ha borrado nada. Si es correcto, sube MAX_FACTURAS_A_BORRAR en la Celda 1; "
                           f"si no, revisa que el DBF de origen esté completo. Primeras: {desaparecen[:5]}")


def avisar_colision_documentos(df, nombre):
    # Solo AVISA. 'documento' = serie + número; si dos series distintas dieran el
    # mismo texto (p. ej. 'A1'+23 y 'A'+123), sus líneas se mezclarían sin error.
    pares = df[['documento', 'CSERIE', 'NNUMFAC']].drop_duplicates()
    choques = pares[pares.duplicated('documento', keep=False)]
    if len(choques) > 0:
        print(f"⚠️ AVISO [{nombre}]: {choques['documento'].nunique()} documentos corresponden a más de una serie/número distinto:")
        print(choques.to_string(index=False))


def cargar_facturacion_clientes():
    facclit_dbf = leer_dbf("Facclit.dbf")
    facclil = leer_dbf("Facclil.dbf")

    # Todas las líneas del DBF, ANTES de descartar ninguna: la última celda
    # del notebook las necesita para cuadrar el total "al céntimo". Cada
    # línea descartada cuenta en UNA sola regla (por eso las máscaras se
    # excluyen entre sí), igual que el filtro de antes.
    facclil_origen = facclil
    m_texto = facclil_origen['CREF'].astype(str).str.contains('Fact|Rectifica', na=False)
    m_cero = facclil_origen['CDETALLE'].isna() & (facclil_origen['NTOTLINEA'] == 0)
    m_vacio = facclil_origen['CREF'].astype(str).str.strip() == ''
    descartes = [
        ('CREF con el texto "Fact" o "Rectifica"', m_texto),
        ('importe 0 y sin detalle', ~m_texto & m_cero),
        ('CREF en blanco', ~m_texto & ~m_cero & m_vacio),
    ]
    facclil = facclil_origen[~m_texto & ~m_cero & ~m_vacio].copy()

    facclit_merge = facclit_dbf[['CSERIE', 'NNUMFAC', 'DFECFAC', 'CCODCLI']]
    df = facclil.merge(facclit_merge, on=['CSERIE', 'NNUMFAC'], how='left')
    df['documento'] = df['CSERIE'] + df['NNUMFAC'].astype(str)
    avisar_colision_documentos(df, 'facturacion_clientes')
    # Fijar el orden ORIGINAL de las lineas ANTES de cualquier
    # merge -- los sucesivos cruces (articulos_bd, respaldo,
    # tipo_operacion_bd) pueden reordenar filas internamente cuando
    # hay coincidencias multiples, y sin esto la huella terminaria
    # comparando lineas en un orden que ya no es el real del DBF
    # (bug real encontrado: DOC-09, misma linea cref+cprop resuelta
    # bien la primera vez y como None la segunda, por desorden).
    df['_orden_original'] = range(len(df))
    # El orden NATURAL de lectura del DBF ya basta como posición de
    # cada línea dentro de su factura — no hace falta guardar ningún
    # número explícito, ni como columna en la tabla ni en ningún sitio.

    clientes_bd = pd.read_sql("SELECT id_cliente, ccodcli FROM clientes", engine)
    df['CCODCLI'] = df['CCODCLI'].astype(str).str.zfill(6)
    df = df.merge(clientes_bd, left_on='CCODCLI', right_on='ccodcli', how='left')

    df['CPROP1'] = df['CPROP1'].fillna('')
    articulos_bd = pd.read_sql("SELECT cref, cprop, id_inmueble FROM articulos_inmuebles", engine)
    df = df.merge(
        articulos_bd, left_on=['CREF', 'CPROP1'], right_on=['cref', 'cprop'], how='left', indicator='_en_puente_inm'
    )
    respaldo = articulos_bd[articulos_bd['cprop'] == ''][['cref', 'id_inmueble']].rename(
        columns={'id_inmueble': 'id_inmueble_respaldo'}
    )
    df = df.merge(respaldo, left_on='CREF', right_on='cref', how='left', suffixes=('', '_resp'), indicator='_tiene_respaldo')
    articulos_de_propiedades = set(articulos_bd.loc[articulos_bd['id_inmueble'].notna(), 'cref'])
    # Antes, CUALQUIER comodín en blanco silenciaba etiquetas
    # desconocidas del mismo artículo (bug real: una unidad en un artículo
    # de una promoción con ~200 unidades mapeadas se habría perdido sin
    # avisar). Ahora, un comodín en blanco solo cubre en silencio las
    # líneas que de verdad llegan sin CPROP -- una etiqueta nueva en un
    # artículo que SÍ reparte por inmueble (tiene alguna fila con
    # id_inmueble) genera aviso, como cualquier combinación sin
    # resolver. Los artículos generales de empresa (sin ninguna fila
    # con inmueble) siguen silenciados por completo, como siempre.
    usa_respaldo = df['id_inmueble'].isna() & (df['_tiene_respaldo'] == 'both') & (
        df['id_inmueble_respaldo'].notna() | ~df['CREF'].isin(articulos_de_propiedades))
    df.loc[usa_respaldo, 'id_inmueble'] = df.loc[usa_respaldo, 'id_inmueble_respaldo']
    df.loc[usa_respaldo, '_en_puente_inm'] = 'both'

    # BUG encontrado hoy: el merge anterior (articulos_inmuebles) ya
    # dejó una columna 'cref' en df, pero SOLO tiene valor para las
    # filas con coincidencia EXACTA -- para el resto (incluidas las
    # resueltas por el respaldo genérico) queda en blanco. Cruzar por
    # esa columna 'cref' contaminada, en vez de por el CREF original,
    # perdía tipo_operacion silenciosamente para muchas filas.
    tipo_operacion_bd = pd.read_sql("SELECT cref, tipo_operacion FROM articulos_tipo_operacion", engine)
    tipo_operacion_bd = tipo_operacion_bd.rename(columns={'cref': 'cref_tipo'})
    df = df.merge(tipo_operacion_bd, left_on='CREF', right_on='cref_tipo', how='left', suffixes=('', '_op'), indicator='_en_puente_tipo')

    CUADRE['clientes'] = {
        'fichero': 'Facclil.dbf',
        'origen_n': len(facclil_origen),
        'origen_cent': int(a_centimos(facclil_origen['NTOTLINEA']).sum()),
        'descartes': [(t, int(m.sum()), int(a_centimos(facclil_origen.loc[m, 'NTOTLINEA']).sum())) for t, m in descartes],
        'lineas': df[['documento', 'CREF', 'CPROP1', 'id_inmueble', 'tipo_operacion', 'NTOTLINEA', 'DFECFAC']]
                  .rename(columns={'DFECFAC': 'fecha'}).copy(),
    }

    sin_cliente = df['id_cliente'].isna().sum()
    if sin_cliente > 0:
        print(f"⚠️ AVISO: {sin_cliente} líneas de facturacion_clientes SIN id_cliente resuelto:")
        print(df[df['id_cliente'].isna()][['documento', 'CCODCLI']].drop_duplicates().to_string(index=False))

    nuevo_df = df[['documento', '_orden_original', 'DFECFAC', 'id_cliente', 'id_inmueble', 'tipo_operacion', 'NTOTLINEA']].copy()
    nuevo_df.columns = ['documento', '_orden_original', 'fecha', 'id_cliente', 'id_inmueble', 'tipo_operacion', 'importe_linea']

    # ─── Comparar la FACTURA COMPLETA (huella de todas sus líneas juntas) ───
    # Sin ningún campo de origen que identifique "esta línea concreta"
    # de forma estable (Facclil.dbf no tiene ningún NLINEA/NORDEN
    # propio), no se puede actualizar línea a línea con seguridad
    # total — si se borra una línea del medio, las siguientes se
    # desplazan de posición. Solución automática y sin ambigüedad:
    # comparar el conjunto de líneas de la factura como un bloque único
    # (mantenidas en su orden natural de lectura, sin numerarlas
    # explícitamente ni guardar nada), y si no coincide exactamente,
    # borrar la factura entera y reinsertarla tal cual está ahora.
    # PostgreSQL devuelve NUMERIC como Decimal, no como float -- sin
    # forzar un formato consistente, "300.0" (del DBF) y "300.00" (de
    # la base de datos) parecerían textos distintos en la huella,
    # aunque sean el mismo importe. Redondear a 2 decimales y formatear
    # siempre igual (":.2f") antes de comparar.
    def _fmt(valor):
        return f"{float(valor):.2f}" if pd.notna(valor) else "NULL"

    def _fmt_texto(valor):
        # None (Python) vs NaN (numpy, tras leer de vuelta de
        # PostgreSQL en una columna con huecos) se imprimen
        # distinto ("None" vs "nan") aunque signifiquen lo mismo
        # -- mismo motivo que con importes y fechas, normalizar
        # explicitamente a un unico marcador fijo.
        return str(valor) if pd.notna(valor) else "NULL"

    def _fmt_fecha(valor):
        # Normaliza a "AAAA-MM-DD", sin importar el tipo exacto con
        # el que llegue la fecha (Timestamp de pandas o date de
        # PostgreSQL) -- mismo motivo que en proveedores.
        return pd.Timestamp(valor).strftime('%Y-%m-%d') if pd.notna(valor) else "NULL"

    def huella_nuevo(d):
        # Una cadena por factura con todas sus líneas, en su orden original del DBF.
        # Vectorizado: da exactamente el mismo texto que el bucle línea a línea de antes.
        return huellas_por_documento(d, '_orden_original')

    def huella_existente(d):
        # id_linea ascendente reconstruye el orden original de inserción
        return huellas_por_documento(d, 'id_linea')

    huella_nueva = huella_nuevo(nuevo_df).rename('huella_nueva')

    existente_df = pd.read_sql(
        "SELECT id_linea, documento, fecha, id_cliente, id_inmueble, tipo_operacion, importe_linea FROM facturacion_clientes",
        engine
    )
    huella_actual = huella_existente(existente_df).rename('huella_actual') if len(existente_df) > 0 else pd.Series(dtype=str, name='huella_actual')

    comparacion = pd.concat([huella_nueva, huella_actual], axis=1)
    documentos_existian = set(existente_df['documento'].unique()) if len(existente_df) > 0 else set()
    documentos_a_reemplazar = comparacion[comparacion['huella_nueva'] != comparacion['huella_actual']].index.tolist()
    documentos_a_reemplazar = sorted(set(documentos_a_reemplazar) | set(documentos_con_otro_tercero(nuevo_df, existente_df)))
    controlar_desapariciones(documentos_a_reemplazar, documentos_existian, nuevo_df)

    # CORREGIDO: antes, si ningún documento cambiaba, la función salía
    # aquí sin comprobar si hay combinaciones CREF/CPROP sin resolver
    # en la tabla puente -- "avisos: 0" no significaba "todo resuelto",
    # significaba "no se ha llegado a comprobar" (bug real: un mismo
    # aviso de 78 combinaciones desaparecía de una ejecución a la
    # siguiente solo porque no cambió ningún documento, sin que el
    # problema de fondo se hubiera resuelto). Ahora se comprueba
    # siempre, cambie o no cambie algún documento.
    sin_revisar = df[
        (df['_en_puente_inm'] == 'left_only') | (df['_en_puente_tipo'] == 'left_only')
    ][['CREF', 'CPROP1']].drop_duplicates()

    if len(documentos_a_reemplazar) == 0:
        informe("facturacion_clientes", 0, 0, len(sin_revisar))
        if len(sin_revisar) > 0:
            print(sin_revisar.to_string(index=False))
        return

    documentos_nuevos = [d for d in documentos_a_reemplazar if d not in documentos_existian]
    documentos_modificados = [d for d in documentos_a_reemplazar if d in documentos_existian]

    # Contar cuántas líneas ANTIGUAS se van a borrar, ANTES de
    # borrarlas -- puede no coincidir con las líneas nuevas que se
    # insertan a continuación (si la factura ahora tiene más o menos
    # líneas que antes)
    lineas_eliminadas = len(existente_df[existente_df['documento'].isin(documentos_modificados)])

    # Borrar e insertar en UNA sola transacción: o se hacen las dos cosas o ninguna
    # (antes, si la inserción fallaba tras confirmarse el borrado, las facturas
    # modificadas desaparecían de la base de datos hasta la siguiente ejecución).
    # El orden de inserción se fuerza al orden original del DBF: de él depende que
    # id_linea reconstruya ese orden más adelante (la huella se compara así).
    tabla_final = nuevo_df[nuevo_df['documento'].isin(documentos_a_reemplazar)].sort_values('_orden_original').drop(columns=['_orden_original'])
    with engine.begin() as conn:
        if len(documentos_modificados) > 0:
            conn.execute(
                text("DELETE FROM facturacion_clientes WHERE documento IN :docs").bindparams(docs=tuple(documentos_modificados))
            )
        tabla_final.to_sql('facturacion_clientes', conn, if_exists='append', index=False)


    lineas_nuevas = len(tabla_final[tabla_final['documento'].isin(documentos_nuevos)])
    lineas_modificadas = len(tabla_final[tabla_final['documento'].isin(documentos_modificados)])
    informe("facturacion_clientes", lineas_nuevas, lineas_modificadas, len(sin_revisar), lineas_eliminadas)
    if len(sin_revisar) > 0:
        print(sin_revisar.to_string(index=False))


def cargar_facturacion_proveedores():
    facprot_dbf = leer_dbf("Facprot.dbf")
    facprol_dbf = leer_dbf("Facprol.dbf")
    facprot_dbf['DSUFECFAC'] = pd.to_datetime(facprot_dbf['DSUFECFAC'], errors='coerce')

    # Todas las líneas del DBF, ANTES de descartar ninguna (ver clientes)
    facprol_origen = facprol_dbf
    m_texto = facprol_origen['CREF'].astype(str).str.contains('Fact|Rectifica', na=False)
    m_vacio = facprol_origen['CREF'].astype(str).str.strip() == ''
    if 'CDETALLE' in facprol_origen.columns:
        m_cero = facprol_origen['CDETALLE'].isna() & (facprol_origen['NTOTLINEA'] == 0)
    else:
        m_cero = pd.Series(False, index=facprol_origen.index)
    descartes = [
        ('CREF con el texto "Fact" o "Rectifica"', m_texto),
        ('CREF en blanco', ~m_texto & m_vacio),
        ('importe 0 y sin detalle', ~m_texto & ~m_vacio & m_cero),
    ]
    facprol_dbf = facprol_origen[~m_texto & ~m_vacio & ~m_cero].copy()

    facprot_merge = facprot_dbf[['CSERIE', 'NNUMFAC', 'CSUFAC', 'DSUFECFAC', 'CCODPRO']]
    df = facprol_dbf[['CSERIE', 'NNUMFAC', 'CREF', 'CPROP1', 'NTOTLINEA']].merge(
        facprot_merge, on=['CSERIE', 'NNUMFAC'], how='left'
    )
    df['documento'] = df['CSERIE'] + df['NNUMFAC'].astype(str)
    avisar_colision_documentos(df, 'facturacion_proveedores')
    # Fijar el orden ORIGINAL de las lineas ANTES de cualquier
    # merge -- los sucesivos cruces (articulos_bd, respaldo,
    # tipo_operacion_bd) pueden reordenar filas internamente cuando
    # hay coincidencias multiples, y sin esto la huella terminaria
    # comparando lineas en un orden que ya no es el real del DBF
    # (bug real encontrado: DOC-09, misma linea cref+cprop resuelta
    # bien la primera vez y como None la segunda, por desorden).
    df['_orden_original'] = range(len(df))

    proveedores_bd = pd.read_sql("SELECT id_proveedor, ccodpro FROM proveedores", engine)
    df['CCODPRO'] = df['CCODPRO'].astype(str).str.zfill(6)
    df = df.merge(proveedores_bd, left_on='CCODPRO', right_on='ccodpro', how='left')

    df['CPROP1'] = df['CPROP1'].fillna('')
    articulos_bd = pd.read_sql("SELECT cref, cprop, id_inmueble FROM articulos_inmuebles", engine)
    df = df.merge(
        articulos_bd, left_on=['CREF', 'CPROP1'], right_on=['cref', 'cprop'], how='left', indicator='_en_puente_inm'
    )
    respaldo = articulos_bd[articulos_bd['cprop'] == ''][['cref', 'id_inmueble']].rename(
        columns={'id_inmueble': 'id_inmueble_respaldo'}
    )
    df = df.merge(respaldo, left_on='CREF', right_on='cref', how='left', suffixes=('', '_resp'), indicator='_tiene_respaldo')
    articulos_de_propiedades = set(articulos_bd.loc[articulos_bd['id_inmueble'].notna(), 'cref'])
    # Antes, CUALQUIER comodín en blanco silenciaba etiquetas
    # desconocidas del mismo artículo (bug real: una unidad en un artículo
    # de una promoción con ~200 unidades mapeadas se habría perdido sin
    # avisar). Ahora, un comodín en blanco solo cubre en silencio las
    # líneas que de verdad llegan sin CPROP -- una etiqueta nueva en un
    # artículo que SÍ reparte por inmueble (tiene alguna fila con
    # id_inmueble) genera aviso, como cualquier combinación sin
    # resolver. Los artículos generales de empresa (sin ninguna fila
    # con inmueble) siguen silenciados por completo, como siempre.
    usa_respaldo = df['id_inmueble'].isna() & (df['_tiene_respaldo'] == 'both') & (
        df['id_inmueble_respaldo'].notna() | ~df['CREF'].isin(articulos_de_propiedades))
    df.loc[usa_respaldo, 'id_inmueble'] = df.loc[usa_respaldo, 'id_inmueble_respaldo']
    df.loc[usa_respaldo, '_en_puente_inm'] = 'both'

    # BUG encontrado hoy: el merge anterior (articulos_inmuebles) ya
    # dejó una columna 'cref' en df, pero SOLO tiene valor para las
    # filas con coincidencia EXACTA -- para el resto (incluidas las
    # resueltas por el respaldo genérico) queda en blanco. Cruzar por
    # esa columna 'cref' contaminada, en vez de por el CREF original,
    # perdía tipo_operacion silenciosamente para muchas filas.
    tipo_operacion_bd = pd.read_sql("SELECT cref, tipo_operacion FROM articulos_tipo_operacion", engine)
    tipo_operacion_bd = tipo_operacion_bd.rename(columns={'cref': 'cref_tipo'})
    df = df.merge(tipo_operacion_bd, left_on='CREF', right_on='cref_tipo', how='left', suffixes=('', '_op'), indicator='_en_puente_tipo')

    CUADRE['proveedores'] = {
        'fichero': 'Facprol.dbf',
        'origen_n': len(facprol_origen),
        'origen_cent': int(a_centimos(facprol_origen['NTOTLINEA']).sum()),
        'descartes': [(t, int(m.sum()), int(a_centimos(facprol_origen.loc[m, 'NTOTLINEA']).sum())) for t, m in descartes],
        'lineas': df[['documento', 'CREF', 'CPROP1', 'id_inmueble', 'tipo_operacion', 'NTOTLINEA', 'DSUFECFAC']]
                  .rename(columns={'DSUFECFAC': 'fecha'}).copy(),
    }

    sin_proveedor = df['id_proveedor'].isna().sum()
    if sin_proveedor > 0:
        print(f"⚠️ AVISO: {sin_proveedor} líneas de facturacion_proveedores SIN id_proveedor resuelto:")
        print(df[df['id_proveedor'].isna()][['documento', 'CCODPRO']].drop_duplicates().to_string(index=False))

    nuevo_df = df[['documento', '_orden_original', 'CSUFAC', 'DSUFECFAC', 'id_proveedor', 'id_inmueble', 'tipo_operacion', 'NTOTLINEA']].copy()
    nuevo_df.columns = ['documento', '_orden_original', 'numero_factura_proveedor', 'fecha', 'id_proveedor', 'id_inmueble', 'tipo_operacion', 'importe_linea']

    # Misma filosofía que en clientes — sin numerar nada, usando el
    # orden natural del DBF y el id_linea existente como referencia de
    # orden en la base de datos. Ver comentario detallado en
    # cargar_facturacion_clientes().
    # Mismo motivo que en clientes: forzar formato numerico consistente
    # (Decimal de PostgreSQL vs float del DBF darian textos distintos
    # para el mismo importe si no se normaliza antes de comparar)
    def _fmt(valor):
        return f"{float(valor):.2f}" if pd.notna(valor) else "NULL"

    def _fmt_texto(valor):
        # None (Python) vs NaN (numpy, tras leer de vuelta de
        # PostgreSQL en una columna con huecos) se imprimen
        # distinto ("None" vs "nan") aunque signifiquen lo mismo
        # -- mismo motivo que con importes y fechas, normalizar
        # explicitamente a un unico marcador fijo.
        return str(valor) if pd.notna(valor) else "NULL"

    def _fmt_fecha(valor):
        # Normaliza a "AAAA-MM-DD", sin importar si llega como
        # Timestamp de pandas (con hora incluida) o como date simple
        # de PostgreSQL -- si no, "2013-10-28 00:00:00" (Timestamp)
        # y "2013-10-28" (date) parecerian el mismo dia pero
        # compararian como texto distinto.
        return pd.Timestamp(valor).strftime('%Y-%m-%d') if pd.notna(valor) else "NULL"

    def huella_nuevo(d):
        # Una cadena por factura con todas sus líneas, en su orden original del DBF.
        # Vectorizado: da exactamente el mismo texto que el bucle línea a línea de antes.
        return huellas_por_documento(d, '_orden_original', 'numero_factura_proveedor')

    def huella_existente(d):
        # id_linea ascendente reconstruye el orden original de inserción
        return huellas_por_documento(d, 'id_linea', 'numero_factura_proveedor')

    huella_nueva = huella_nuevo(nuevo_df).rename('huella_nueva')

    existente_df = pd.read_sql(
        "SELECT id_linea, documento, numero_factura_proveedor, fecha, id_proveedor, id_inmueble, tipo_operacion, importe_linea "
        "FROM facturacion_proveedores",
        engine
    )
    huella_actual = huella_existente(existente_df).rename('huella_actual') if len(existente_df) > 0 else pd.Series(dtype=str, name='huella_actual')

    comparacion = pd.concat([huella_nueva, huella_actual], axis=1)
    documentos_existian = set(existente_df['documento'].unique()) if len(existente_df) > 0 else set()
    documentos_a_reemplazar = comparacion[comparacion['huella_nueva'] != comparacion['huella_actual']].index.tolist()
    documentos_a_reemplazar = sorted(set(documentos_a_reemplazar) | set(documentos_con_otro_tercero(nuevo_df, existente_df)))
    controlar_desapariciones(documentos_a_reemplazar, documentos_existian, nuevo_df)

    # CORREGIDO: antes, si ningún documento cambiaba, la función salía
    # aquí sin comprobar si hay combinaciones CREF/CPROP sin resolver
    # en la tabla puente -- "avisos: 0" no significaba "todo resuelto",
    # significaba "no se ha llegado a comprobar" (bug real: un mismo
    # aviso de 78 combinaciones desaparecía de una ejecución a la
    # siguiente solo porque no cambió ningún documento, sin que el
    # problema de fondo se hubiera resuelto). Ahora se comprueba
    # siempre, cambie o no cambie algún documento.
    sin_revisar = df[
        (df['_en_puente_inm'] == 'left_only') | (df['_en_puente_tipo'] == 'left_only')
    ][['CREF', 'CPROP1']].drop_duplicates()

    if len(documentos_a_reemplazar) == 0:
        informe("facturacion_proveedores", 0, 0, len(sin_revisar))
        if len(sin_revisar) > 0:
            print(sin_revisar.to_string(index=False))
        return

    documentos_nuevos = [d for d in documentos_a_reemplazar if d not in documentos_existian]
    documentos_modificados = [d for d in documentos_a_reemplazar if d in documentos_existian]

    # Mismo motivo que en clientes: contar las líneas antiguas antes
    # de borrarlas, por si la factura cambia de número de líneas
    lineas_eliminadas = len(existente_df[existente_df['documento'].isin(documentos_modificados)])

    # Borrar e insertar en UNA sola transacción: o se hacen las dos cosas o ninguna
    # (antes, si la inserción fallaba tras confirmarse el borrado, las facturas
    # modificadas desaparecían de la base de datos hasta la siguiente ejecución).
    # El orden de inserción se fuerza al orden original del DBF: de él depende que
    # id_linea reconstruya ese orden más adelante (la huella se compara así).
    tabla_final = nuevo_df[nuevo_df['documento'].isin(documentos_a_reemplazar)].sort_values('_orden_original').drop(columns=['_orden_original'])
    with engine.begin() as conn:
        if len(documentos_modificados) > 0:
            conn.execute(
                text("DELETE FROM facturacion_proveedores WHERE documento IN :docs").bindparams(docs=tuple(documentos_modificados))
            )
        tabla_final.to_sql('facturacion_proveedores', conn, if_exists='append', index=False)


    lineas_nuevas = len(tabla_final[tabla_final['documento'].isin(documentos_nuevos)])
    lineas_modificadas = len(tabla_final[tabla_final['documento'].isin(documentos_modificados)])
    informe("facturacion_proveedores", lineas_nuevas, lineas_modificadas, len(sin_revisar), lineas_eliminadas)
    if len(sin_revisar) > 0:
        print(sin_revisar.to_string(index=False))


## Celda 6 — Ejecutar carga documental, en orden

Llama a las funciones anteriores en el orden que exige sus
dependencias: clientes/proveedores primero (para que ya existan al
resolver facturación), después contratos, después facturación.

**Ya no hace falta ninguna reconciliación aparte**: con el diseño
actual de `cargar_facturacion_clientes()`/`proveedores()` (que
recalcula todas las líneas cada vez, comparando por huella completa
contra lo ya guardado), cualquier cambio en las tablas puente
(`articulos_inmuebles`/`articulos_tipo_operacion`) se detecta y
corrige solo, en la siguiente ejecución — el mismo mecanismo que ya
detecta cambios en las propias facturas. La función de reconciliación
que existía antes se eliminó tras confirmar que, además de
redundante, corrompía datos (emparejaba por `documento`+
`importe_linea`, una clave no única cuando dos líneas de una misma
factura cuestan lo mismo — caso real encontrado: `DOC-09`).


In [ ]:
# Orden obligatorio: clientes/proveedores antes de contratos y
# facturación (que necesitan resolver id_cliente/id_proveedor);
# contratos antes de agrupar_contratos (que parte de lo ya cargado)
cargar_inmuebles()
cargar_articulos_inmuebles()
cargar_articulos_tipo_operacion()
cargar_clientes()
cargar_proveedores()
cargar_contratos()
agrupar_contratos()
cargar_facturacion_clientes()
cargar_facturacion_proveedores()


# Parte 2 — Reconciliación de cobros y pagos

Depende de que `facturacion_clientes`/`facturacion_proveedores` ya
estén cargadas (Parte 1, arriba) — reutiliza el mismo `engine` y
`leer_dbf`/`informe` ya definidos en la Celda 1.

**El objetivo**: saber, para cada factura, cuánto se ha
cobrado/pagado de verdad y cuánto queda pendiente — no solo sumando
los pagos de `Pagdev`, sino siguiendo fielmente cómo FacturaPlus
gestiona las compensaciones entre recibos, incluidas las
rectificativas (notas de abono).

**Idea central**: la unidad real no es "la factura", es "el recibo".
Una factura puede tener varios recibos (uno por plazo), y cada
recibo puede tener su dinero real en **otro** recibo distinto (de la
misma factura o de otra), mediante el campo `CMATRIZ`. Todo el
algoritmo trabaja a ese nivel de detalle.


## Celda 7 — Funciones compartidas de compensación de recibos

**Reescritura completa** tras encontrar, en la misma tarde, varios
fallos sutiles causados por tener DOS cálculos de "grupo de
compensación" separados (uno para el cobro real, otro para el ajuste
nominal de rectificativas) — un arreglo aplicado a uno no llegaba al
otro (así se coló el bug de `DOC-02`, que solo afectaba a uno de los
dos cálculos).

### `calcular_raiz_recibos(recibos)`

Función **única y compartida**: encuentra la "raíz" de cada recibo
(siguiendo `CMATRIZ` recursivamente), incluido el arreglo del formato
sin espacio entre número de factura y número de recibo (`DOC-02`). La
usan tanto `reconciliar_cobros()` como `ajustar_por_grupos_
compensacion()`, para no repetir la misma lógica dos veces con el
riesgo de que un arreglo solo llegue a una de las dos copias.

### `reconciliar_cobros(pagdev, recibos)`

Calcula cuánto se ha cobrado realmente, siguiendo la raíz de cada
recibo hasta el dinero real en `Pagdev`, y repartiéndolo entre el
grupo. Ahora usa `calcular_raiz_recibos()` en vez de su propia copia.

### `ajustar_por_grupos_compensacion(...)`

**Sustituye a los antiguos Mecanismos A/B/C y al mecanismo de "grupo
saldado"** — una única lógica coherente para cualquier grupo de
recibos conectados por `CMATRIZ`, sea cual sea su tamaño:

- Solo compensa cuando la raíz del grupo y el resto tienen **signos
  opuestos** (una resta real) — si comparten signo, sumarlos solo
  infla la deuda en vez de cancelarla (caso real: `DOC-21`→`DOC-20`, dos
  rectificativas, se dejan independientes).
- Trabaja por **línea de recibo**, nunca por el total de un documento
  completo — un documento puede tener varios recibos, y solo algunos
  pertenecer a un grupo concreto (caso real: `DOC-10`).
- Si el grupo suma `0€`, todo queda resuelto. Si no, la raíz absorbe
  el ajuste (topada para no cruzar de signo), y el sobrante se
  reparte proporcionalmente entre los demás miembros del grupo,
  quedando visible como su propio pendiente.

Probado con los 7 casos reales encontrados a lo largo de la sesión
(`DOC-13`/`DOC-14`, `DOC-16`/`DOC-17`/`DOC-18`, un grupo de un proveedor de 6
miembros, `DOC-21`/`DOC-20`, `DOC-02`, `DOC-10`/`DOC-11`/`DOC-12`, `DOC-14`+`DOC-15`→
`DOC-01`) antes de integrarlo.

### `limitar_sobrante(...)`

Sin cambios de diseño, salvo el arreglo de hoy: decide en qué
dirección recortar el sobrante mirando el signo del **cobro real**,
no el del importe de la factura (caso real: `DOC-04`, `DOC-05`).

### `concentrar_cobro_en_raiz(...)`

Para los grupos que SÍ se compensaron nominalmente (mezcla real de
signos), el dinero real cobrado se concentra también en el documento
raíz, en vez de dejar que se reparta por fecha de vencimiento de forma
independiente al ajuste nominal. Sin esto, ambos cálculos podían
discrepar sobre quién se queda con qué parte del dinero real (caso
real: un caso de dos clientes, grupo de 6 documentos, donde el reparto por
fecha y el ajuste nominal no coincidían, generando sobrecobros falsos
en unos documentos y un pendiente sin cubrir en la raíz). Los grupos
normales de domiciliación conjunta (sin ninguna rectificativa) nunca
pasan por aquí — siguen repartiéndose por fecha exactamente como
siempre.


In [ ]:
def calcular_raiz_recibos(recibos, col_serie='CSERIE', col_numero='NNUMFAC'):
    # Función ÚNICA y compartida para encontrar la "raíz" de cada
    # recibo (siguiendo CMATRIZ recursivamente) -- se usa tanto para
    # repartir el dinero real (reconciliar_cobros) como para ajustar
    # el importe nominal de las rectificativas (ajustar_por_grupos_
    # compensacion). Antes existían DOS implementaciones separadas de
    # esto mismo, y un arreglo aplicado a una no llegaba a la otra
    # (así se coló el bug de DOC-02, que solo afectaba al cálculo usado
    # por reconciliar_cobros, sin llegar al mecanismo de grupos).
    recibos = recibos.copy()
    mapa_matriz = {}
    for _, row in recibos.iterrows():
        if row['CMATRIZ'] not in [None, '']:
            partes = row['CMATRIZ'].split()
            if len(partes) >= 3:
                mapa_matriz[(row[col_serie], row[col_numero], row['CCONTADOR'])] = (partes[0], int(partes[1]), int(partes[2]))
            elif len(partes) == 2:
                serie_destino, token = partes[0], partes[1]
                numero_propio_str = str(row[col_numero])
                # Fallo de formato real (DOC-02): a veces falta el
                # espacio entre NUMERO y CONTADOR, quedando
                # concatenados (ej. "41729" en vez de "417 29") -- si
                # el token empieza por el propio número de esta
                # factura, es un recibo de la MISMA factura, no un
                # documento externo distinto.
                if token.startswith(numero_propio_str) and len(token) > len(numero_propio_str):
                    contador_real = int(token[len(numero_propio_str):])
                    mapa_matriz[(row[col_serie], row[col_numero], row['CCONTADOR'])] = (serie_destino, row[col_numero], contador_real)
                else:
                    mapa_matriz[(row[col_serie], row[col_numero], row['CCONTADOR'])] = (serie_destino, int(token), 1)

    def encontrar_raiz(serie, numero, contador, visitados=None):
        # Protección contra ciclos con "visitados"
        if visitados is None:
            visitados = set()
        clave = (serie, numero, contador)
        if clave in visitados:
            return clave
        visitados.add(clave)
        if clave in mapa_matriz:
            return encontrar_raiz(*mapa_matriz[clave], visitados)
        return clave

    recibos['raiz'] = recibos.apply(
        lambda r: encontrar_raiz(r[col_serie], r[col_numero], r['CCONTADOR']), axis=1
    )
    return recibos


import pandas as pd

import pandas as pd

def reconciliar_cobros(pagdev, recibos, col_serie='CSERIE', col_numero='NNUMFAC'):
    pagdev_ultimo = pagdev.sort_values('NESTADOANT').groupby([col_serie, col_numero, 'CCONTADOR']).last().reset_index()
    recibos_clave = calcular_raiz_recibos(recibos[[col_serie, col_numero, 'CCONTADOR', 'NIMPORTE', 'CMATRIZ', 'DFECVCTO']].copy(), col_serie, col_numero)
    recibos_clave['raiz_serie'] = recibos_clave['raiz'].apply(lambda x: x[0])
    recibos_clave['raiz_numero'] = recibos_clave['raiz'].apply(lambda x: x[1])
    recibos_clave['raiz_contador'] = recibos_clave['raiz'].apply(lambda x: x[2])
    pagdev_ultimo_r = pagdev_ultimo.rename(columns={col_serie:'raiz_serie',col_numero:'raiz_numero','CCONTADOR':'raiz_contador','DFECHA':'fecha_pago_raiz','NIMPORTE':'importe_pago_raiz'})[['raiz_serie','raiz_numero','raiz_contador','fecha_pago_raiz','importe_pago_raiz']]
    recibos_clave = recibos_clave.merge(pagdev_ultimo_r, on=['raiz_serie','raiz_numero','raiz_contador'], how='left')
    resultado_cobros = []
    for (rs,rn,rc), grupo in recibos_clave.groupby(['raiz_serie','raiz_numero','raiz_contador']):
        if len(grupo) > 1:       # un grupo de un solo recibo ya está ordenado: ahorrarse el sort_values (era lo que más tardaba)
            grupo = grupo.sort_values(['DFECVCTO', col_serie, col_numero])
        importe_disponible = grupo['importe_pago_raiz'].iloc[0]
        fecha_pago = grupo['fecha_pago_raiz'].iloc[0]
        if pd.isna(importe_disponible):
            continue
        raiz_tupla_grupo = (rs, rn, rc)  # incluye CCONTADOR -- antes se
        # perdia, confundiendo dos raices distintas del MISMO
        # documento (bug real: DOC-19 recibo 1, independiente, y DOC-19
        # recibo 3, la raiz del grupo con DOC-05, se identificaban con
        # el mismo texto "DOC-19" y se mezclaban entre si)
        restante = importe_disponible
        for _, recibo in grupo.iterrows():
            if abs(restante) <= 0.005:
                break
            documento = f"{recibo[col_serie]}{recibo[col_numero]}"
            monto_recibo = recibo['NIMPORTE']
            if restante * monto_recibo < 0:
                continue
            aplicado = min(restante, monto_recibo) if restante >= 0 else max(restante, monto_recibo)
            if abs(aplicado) > 0.005:
                # 'raiz_documento' identifica de qué grupo (raíz)
                # concreto vino este cobro -- imprescindible para no
                # confundirlo con otro cobro independiente del MISMO
                # documento que viniera de un recibo distinto, sin
                # relación con este grupo (bug real: DOC-08 tiene un
                # recibo en este grupo y otro totalmente aparte, con
                # su propio pago independiente).
                resultado_cobros.append({'documento':documento,'fecha_cobro':fecha_pago,'importe_cobro':round(aplicado,2),'raiz_tupla':raiz_tupla_grupo})
            restante -= aplicado
    resultado_df = pd.DataFrame(resultado_cobros)
    if len(resultado_df)==0:
        return resultado_df
    resultado_df['numero_cuota'] = resultado_df.groupby('documento').cumcount()+1
    return resultado_df[['documento','numero_cuota','fecha_cobro','importe_cobro','raiz_tupla']]


def ajustar_por_grupos_compensacion(recibos_con_raiz, importe_por_documento, nombre_fac):
    # Sustituye a los antiguos Mecanismos B/C y al mecanismo de "grupo
    # saldado" -- una única lógica coherente, que trata cualquier
    # grupo de recibos conectados por CMATRIZ de la misma forma,
    # sea cual sea su tamaño o el signo de sus miembros.
    #
    # Los grupos se identifican por la RAÍZ COMPLETA (serie, número,
    # contador) -- nunca solo por el nombre del documento, porque un
    # mismo documento puede tener varios recibos que sean, cada uno,
    # la raíz de un grupo COMPLETAMENTE DISTINTO (bug real: DOC-19
    # recibo 1, independiente, y DOC-19 recibo 3, raíz del grupo con
    # DOC-05 -- confundir ambos por compartir nombre de documento
    # mezclaba su dinero real sin ninguna relación entre sí).
    #
    # Para cada grupo (recibos que comparten la misma raíz):
    #   - El documento DUEÑO de la raíz es la "base" del grupo.
    #   - Se suma el aporte de TODOS los demás miembros (por línea de
    #     recibo, nunca por el total de un documento completo).
    #   - Si la suma total del grupo da 0€, todo queda resuelto sin
    #     sobrante. Si no, la base absorbe el ajuste, topada para no
    #     cruzar de signo.
    #   - Cada documento no-base se reduce por la suma de SUS PROPIAS
    #     líneas de recibo en el grupo.
    #
    # Devuelve: reduccion_por_documento (documento -> cuánto restar de
    # su importe nominal), grupos_procesados (raíz completa ->
    # documentos del grupo) e importes_finales_grupo (raíz completa ->
    # {documento: su importe nominal ya ajustado}), necesario para que
    # concentrar_cobro_en_raiz() sepa a quién del grupo concentrar el
    # dinero real.
    reduccion_por_documento = {}
    avisos_sobrante = []
    grupos_procesados = {}
    importes_finales_grupo = {}

    for raiz, grupo in recibos_con_raiz.groupby('raiz'):
        documentos_unicos = list(dict.fromkeys(grupo['documento']))
        if len(documentos_unicos) <= 1:
            continue

        serie_raiz, numero_raiz, contador_raiz = raiz
        documento_raiz = f"{serie_raiz}{numero_raiz}"

        es_raiz = (
            (grupo['CSERIE'] == serie_raiz) & (grupo['NNUMFAC'] == numero_raiz) & (grupo['CCONTADOR'] == contador_raiz)
        )
        importe_base = grupo.loc[es_raiz, 'NIMPORTE'].sum()
        otros_nimportes = grupo.loc[~es_raiz, 'NIMPORTE']
        ajuste_otros = otros_nimportes.sum()

        if importe_base >= 0:
            hay_signo_opuesto = (otros_nimportes < 0).any()
        else:
            hay_signo_opuesto = (otros_nimportes > 0).any()
        if not hay_signo_opuesto:
            continue

        suma_grupo = importe_base + ajuste_otros

        if abs(suma_grupo) <= 0.01:
            base_final = 0.0
        elif importe_base >= 0:
            base_final = max(suma_grupo, 0.0)
        else:
            base_final = min(suma_grupo, 0.0)

        sobrante = suma_grupo - base_final
        if abs(sobrante) > 0.01:
            avisos_sobrante.append((documentos_unicos, documento_raiz, sobrante))

        grupos_procesados[raiz] = documentos_unicos
        importes_finales_grupo_actual = {}

        for doc, sub in grupo.groupby('documento'):
            if doc == documento_raiz:
                reduccion_aplicada = importe_base - base_final
                importes_finales_grupo_actual[doc] = base_final
            else:
                aporte_doc = sub['NIMPORTE'].sum()
                if abs(sobrante) > 0.01 and ajuste_otros != 0:
                    proporcion = aporte_doc / ajuste_otros
                    aporte_doc_aplicado = aporte_doc - (sobrante * proporcion)
                else:
                    aporte_doc_aplicado = aporte_doc
                reduccion_aplicada = aporte_doc_aplicado
                importes_finales_grupo_actual[doc] = aporte_doc - aporte_doc_aplicado
            reduccion_por_documento[doc] = reduccion_por_documento.get(doc, 0) + reduccion_aplicada

        importes_finales_grupo[raiz] = importes_finales_grupo_actual

    # Los avisos NO se imprimen aquí -- en este punto solo sabemos el
    # desajuste NOMINAL, pero el dinero real (concentrar_cobro_en_raiz,
    # más adelante) puede resolverlo del todo (caso real: DOC-19/DOC-05,
    # donde el ajuste manual del contable cubría exactamente el
    # sobrante nominal). Se devuelven para comprobarlos al FINAL,
    # contra el pendiente real definitivo -- así el aviso solo aparece
    # si de verdad sigue habiendo algo sin resolver.
    return reduccion_por_documento, grupos_procesados, importes_finales_grupo, avisos_sobrante


def concentrar_cobro_en_raiz(cobros_df, grupos_procesados, importes_finales_grupo, col_fecha, col_importe):
    # Para cada grupo que SÍ se compensó nominalmente, el dinero real
    # cobrado se concentra en quien de verdad le quede pendiente tras
    # el ajuste nominal -- no siempre es la raíz (caso real:
    # DOC-19/DOC-05, donde el sobrante quedó en DOC-05, no en DOC-19).
    # Se identifica cada grupo por su RAÍZ COMPLETA (serie, número,
    # contador) -- nunca por nombre de documento, para no mezclar
    # dinero de dos raíces distintas que compartan documento (bug
    # real, ver ajustar_por_grupos_compensacion).
    if len(cobros_df) == 0 or len(grupos_procesados) == 0:
        return cobros_df.drop(columns=['raiz_tupla'], errors='ignore')

    filas_a_quitar = []
    filas_nuevas = []
    for raiz, documentos_grupo in grupos_procesados.items():
        filas_grupo = cobros_df[cobros_df['raiz_tupla'] == raiz]
        if len(filas_grupo) == 0:
            continue
        total_cobrado_grupo = round(filas_grupo[col_importe].sum(), 2)
        fecha_representativa = filas_grupo[col_fecha].iloc[0]
        filas_a_quitar.extend(filas_grupo.index.tolist())
        if abs(total_cobrado_grupo) <= 0.005:
            continue

        documento_raiz = f"{raiz[0]}{raiz[1]}"
        remanentes = importes_finales_grupo.get(raiz, {})
        remanentes_no_cero = {d: v for d, v in remanentes.items() if abs(v) > 0.01}

        if len(remanentes_no_cero) == 0:
            destinos = {documento_raiz: total_cobrado_grupo}
        elif len(remanentes_no_cero) == 1:
            doc_destino = next(iter(remanentes_no_cero))
            destinos = {doc_destino: total_cobrado_grupo}
        else:
            suma_remanentes = sum(remanentes_no_cero.values())
            destinos = {
                d: round(total_cobrado_grupo * (v / suma_remanentes), 2)
                for d, v in remanentes_no_cero.items()
            }

        for doc_destino, importe_destino in destinos.items():
            if abs(importe_destino) > 0.005:
                filas_nuevas.append({
                    'documento': doc_destino, 'numero_cuota': 1,
                    col_fecha: fecha_representativa, col_importe: importe_destino,
                    'raiz_tupla': raiz
                })

    cobros_df = cobros_df.drop(index=filas_a_quitar)
    if filas_nuevas:
        cobros_df = pd.concat([cobros_df, pd.DataFrame(filas_nuevas)], ignore_index=True)
    return cobros_df.drop(columns=['raiz_tupla'])


def limitar_sobrante(cobros_df, facturas_df, col_codigo, col_fecha_cobro, col_importe):
    totales = cobros_df.groupby('documento')[col_importe].sum().reset_index()
    totales.columns = ['documento', 'total_cobrado']
    info = facturas_df.merge(totales, on='documento', how='left')
    info['total_cobrado'] = info['total_cobrado'].fillna(0)
    # Usar valor absoluto para detectar el exceso — para un documento
    # con importe NEGATIVO (rectificativa), "cobrado de más" significa
    # un número todavía MÁS negativo, no uno positivo. Comparar solo
    # con .clip(lower=0) sobre la resta directa (sin abs) deja pasar
    # ese caso sin detectarlo (bug encontrado con dos rectificativas de un proveedor).
    info['sobrante'] = (info['total_cobrado'].abs() - info['importe_total_con_iva'].abs()).clip(lower=0).round(2)

    sobrantes_pendientes_decision = []
    for _, fila in info[info['sobrante'] > 0.01].iterrows():
        doc_origen = fila['documento']
        exceso = fila['sobrante']
        filas_doc = cobros_df[cobros_df['documento'] == doc_origen].sort_values(col_fecha_cobro, ascending=False)
        if len(filas_doc) == 0:
            continue
        idx_recorte = filas_doc.index[0]
        # Recortar hacia cero desde el signo del PROPIO COBRO, no del
        # importe de la factura -- cuando una factura queda en
        # exactamente 0€ (por ejemplo, saldada por el mecanismo de
        # grupo), su importe no dice nada sobre en qué dirección está
        # el cobro real asociado, que puede ser negativo aunque la
        # factura sea "0 >= 0" (caso real: DOC-04, DOC-05, DOC-06/DOC-07 --
        # factura en 0€ con un cobro real negativo, donde el signo
        # equivocado duplicaba el problema en vez de corregirlo).
        if fila['total_cobrado'] >= 0:
            cobros_df.loc[idx_recorte, col_importe] -= exceso
        else:
            cobros_df.loc[idx_recorte, col_importe] += exceso
        fecha_sobrante = filas_doc.iloc[0][col_fecha_cobro]
        sobrantes_pendientes_decision.append({
            'documento': doc_origen, col_codigo: fila[col_codigo],
            'importe_sobrante': exceso, 'fecha': fecha_sobrante
        })
    return cobros_df, pd.DataFrame(sobrantes_pendientes_decision)


## Celda 8 — `procesar_incremental()`: la función principal

Se ejecuta una vez para clientes (Celda 9) y otra para proveedores
(Celda 10). Sus pasos, en el orden en que aparecen en el código:

**1. Cálculo base** — lee los tres DBF y llama a `reconciliar_cobros()`.

**2. `documento_original`** — solo para rectificativas (serie `R`),
a partir de `CNUMFACREC`. Es solo una anotación de referencia (según
la guía de Hacienda), sin efecto en ningún cálculo.

**3. Compensación de recibos (Celda 7)** — una única llamada a
`ajustar_por_grupos_compensacion()`, que sustituye a todo lo que antes
eran 3 mecanismos separados (A/B/C) más un mecanismo de "grupo
saldado" aparte. Ver la Celda 7 para el detalle completo.

**4. `tiene_recibo`** — se marca si el documento tiene al menos un
recibo en origen (caso real: `DOC-03`, sin ningún recibo, nunca
resoluble por este método) — la vista lo usa para mostrar "Sin
recibo, no evaluable" en vez de "Pendiente".

**5. Salvaguarda de referencias** — si `documento_original` apunta a
un documento ya absorbido por la compensación de grupo, se deja en
blanco con aviso.

**6-8. Tope general, comparación incremental y escritura** — igual
que antes: `limitar_sobrante()`, comparación contra lo ya guardado
(importe de la factura, total cobrado, fechas y reparto de los cobros,
`documento_original` y `tiene_recibo`), y `DELETE`+`INSERT` solo de lo
afectado.


In [ ]:
def _huella_cobros(d):
    # Una cadena por documento con TODOS sus cobros (fecha:importe, por orden de cuota): si cambia la
    # fecha de un pago, o cómo se reparte entre cuotas, se detecta aunque el total cobrado sea el mismo.
    if len(d) == 0:
        return pd.Series(dtype=str)
    d = d.sort_values(['documento', 'numero_cuota'])
    f = pd.to_datetime(d['fecha'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('NULL')
    i = pd.to_numeric(d['importe'], errors='coerce').astype(float).map(lambda v: 'NULL' if pd.isna(v) else f"{v:.2f}")
    return (f + ':' + i).groupby(d['documento']).agg('||'.join)


def limpiar_cnumfacrec(valor):
    # Interpreta el campo CNUMFACREC de una rectificativa (p.ej.
    # "A/ 123") y devuelve el documento al que corrige ("A123"), o
    # None si está vacío o con un formato irreconocible. Solo se usa
    # aquí, para calcular documento_original.
    if pd.isna(valor) or valor.strip() == '':
        return None
    partes = valor.split('/')
    if len(partes) != 2:
        return None
    serie, numero = partes[0].strip(), partes[1].strip()
    if not numero.isdigit():
        return None
    return f"{serie}{int(numero)}"


def procesar_incremental(nombre_fac, nombre_recibos, nombre_pagdev,
                          tabla_facturas_totales, tabla_cobros,
                          col_codigo_terceros, col_fecha_cobro, col_importe_cobro, col_fecha_factura):
    # La fecha de la factura se guarda en la propia tabla de facturas totales (viene de la CABECERA del DBF).
    # Antes salía de las líneas, y una factura sin líneas (p. ej. con la línea sin artículo, que se descarta)
    # se quedaba sin fecha. Si la tabla aún no tiene la columna, se añade; si ya la tiene, no hace nada.
    with engine.begin() as conn:
        conn.execute(text(f"ALTER TABLE {tabla_facturas_totales} ADD COLUMN IF NOT EXISTS fecha_factura DATE"))

    # ─── Carga y preparación base ───
    fac = leer_dbf(nombre_fac)
    recibos = leer_dbf(nombre_recibos)
    pagdev = leer_dbf(nombre_pagdev)
    fac['documento'] = fac['CSERIE'] + fac['NNUMFAC'].astype(str)
    recibos['documento'] = recibos['CSERIE'] + recibos['NNUMFAC'].astype(str)

    # Documentos que tienen AL MENOS un recibo -- una factura sin
    # ningún recibo nunca podrá resolverse por este método (no hay
    # ningún dato de cobro que consultar, caso real: DOC-03), así que
    # se marca aparte para poder distinguirla de una pendiente normal
    documentos_con_recibo = set(recibos['documento'])

    # --- Cobro real de cada documento (recalculado en memoria, barato) ---
    cobros_df = reconciliar_cobros(pagdev, recibos)
    # reconciliar_cobros() siempre devuelve 'fecha_cobro'/'importe_cobro'
    # como nombres fijos — renombrar a los nombres reales de esta
    # entidad (fecha_pago/importe_pago en proveedores)
    if len(cobros_df) > 0:
        cobros_df = cobros_df.rename(columns={'fecha_cobro': col_fecha_cobro, 'importe_cobro': col_importe_cobro})
    else:
        cobros_df = pd.DataFrame(columns=['documento', 'numero_cuota', col_fecha_cobro, col_importe_cobro])

    # ─── Importe base de cada factura, tal cual viene de origen ───
    facturas_totales_df = fac[['documento', 'NTOTFAC', 'CNUMFACREC', 'CSERIE', col_codigo_terceros, col_fecha_factura]].copy()
    facturas_totales_df.columns = ['documento', 'importe_total_con_iva', 'CNUMFACREC', 'CSERIE', col_codigo_terceros, 'fecha_factura']
    fechas = pd.to_datetime(facturas_totales_df['fecha_factura'], errors='coerce')
    facturas_totales_df['fecha_factura'] = fechas.dt.date.where(fechas.notna(), None)

    # documento_original: solo para rectificativas (CSERIE='R'), nunca
    # para facturas normales. Motivo: algunas facturas normales TAMBIÉN
    # tienen CNUMFACREC relleno, pero como puntero DE VUELTA hacia su
    # propia rectificativa (confirmado con DOC-13, que apunta a DOC-14) —
    # el sentido contrario al que se busca aquí. Es solo una anotación
    # de referencia (según Hacienda), sin efecto en ningún cálculo —
    # el efecto real lo decide el mecanismo de compensación de abajo.
    facturas_totales_df['documento_original'] = None
    es_rectificativa_serie_r = facturas_totales_df['CSERIE'] == 'R'
    facturas_totales_df.loc[es_rectificativa_serie_r, 'documento_original'] = (
        facturas_totales_df.loc[es_rectificativa_serie_r, 'CNUMFACREC'].apply(limpiar_cnumfacrec)
    )
    facturas_totales_df = facturas_totales_df[['documento', 'importe_total_con_iva', 'documento_original', col_codigo_terceros, 'fecha_factura']]
    facturas_totales_df['tiene_recibo'] = facturas_totales_df['documento'].isin(documentos_con_recibo)

    # ─── COMPENSACIÓN DE RECIBOS (mecanismo único, sustituye A/B/C) ───
    # Reescritura completa tras detectar que mantener el cálculo de
    # "cobro real" (reconciliar_cobros) y el de "ajuste nominal por
    # rectificativas" (antes: Mecanismos B/C + mecanismo de grupo) por
    # SEPARADO, cada uno con su propio cálculo de raíz de CMATRIZ,
    # llevaba a inconsistencias: un arreglo aplicado a un cálculo no
    # llegaba al otro (así se coló el bug de DOC-02). Ahora ambos usan
    # la MISMA función compartida (calcular_raiz_recibos, Celda 7).
    recibos_con_raiz = calcular_raiz_recibos(recibos)
    importe_por_documento_base = facturas_totales_df.set_index('documento')['importe_total_con_iva'].to_dict()
    reduccion_por_documento, grupos_procesados, importes_finales_grupo, avisos_sobrante_grupo = ajustar_por_grupos_compensacion(recibos_con_raiz, importe_por_documento_base, nombre_fac)
    for doc, reduccion in reduccion_por_documento.items():
        mask = facturas_totales_df['documento'] == doc
        facturas_totales_df.loc[mask, 'importe_total_con_iva'] -= reduccion

    # Concentrar el dinero real cobrado en quien de verdad le quede
    # pendiente tras el ajuste nominal (no siempre la raíz) -- ver el
    # motivo detallado en concentrar_cobro_en_raiz() (Celda 7). Casos
    # reales que lo hicieron necesario: un caso de dos clientes (sobrante en la
    # raíz) y DOC-19/DOC-05 (sobrante en el "otro", no en la raíz).
    if len(cobros_df) > 0:
        cobros_df = concentrar_cobro_en_raiz(cobros_df, grupos_procesados, importes_finales_grupo, col_fecha_cobro, col_importe_cobro)

    # ─── Salvaguarda de integridad referencial ───
    # documento_original apunta a la misma tabla (autorreferencia) —
    # si el documento al que apuntaba fue absorbido por la
    # compensación de grupo, o cualquier otra inconsistencia, se
    # detecta y se deja en blanco con aviso, en vez de romper la
    # inserción más adelante.
    documentos_validos = set(facturas_totales_df['documento'])
    referencias_invalidas = facturas_totales_df['documento_original'].notna() & ~facturas_totales_df['documento_original'].isin(documentos_validos)
    if referencias_invalidas.sum() > 0:
        print(f"⚠️ AVISO [{nombre_fac}]: {referencias_invalidas.sum()} documento_original apuntando a algo inexistente, se deja en blanco:")
        print(facturas_totales_df[referencias_invalidas][['documento', 'documento_original']].to_string(index=False))
    facturas_totales_df.loc[referencias_invalidas, 'documento_original'] = None

    # ─── Tope general: ninguna factura cobrada por más de su propio importe ───
    cobros_df = cobros_df[cobros_df['documento'].isin(documentos_validos)]
    cobros_df, sobrantes = limitar_sobrante(cobros_df, facturas_totales_df, col_codigo_terceros, col_fecha_cobro, col_importe_cobro)
    if len(sobrantes) > 0:
        print(f"⚠️ AVISO [{nombre_fac}]: {len(sobrantes)} sobrecobros/sobrepagos pendientes de decisión del contable:")
        print(sobrantes.to_string(index=False))

    facturas_totales_df = facturas_totales_df.drop(columns=[col_codigo_terceros])  # tiene_recibo SÍ se conserva

    # ─── Comparar contra lo que YA está en la base de datos ───
    # Esto es lo que hace la carga "incremental" de verdad: se recalcula
    # todo en memoria (barato), pero solo se ESCRIBE lo que cambió de
    # verdad respecto a lo ya guardado — el resto de la base de datos
    # queda completamente intacto.
    nuevo_total_cobrado = cobros_df.groupby('documento')[col_importe_cobro].sum().reset_index()
    nuevo_total_cobrado.columns = ['documento', 'total_cobrado_nuevo']
    calculado = facturas_totales_df.merge(nuevo_total_cobrado, on='documento', how='left')
    calculado['total_cobrado_nuevo'] = calculado['total_cobrado_nuevo'].fillna(0).round(2)

    # Comprobar AHORA (con el pendiente real ya definitivo, tras
    # concentrar_cobro_en_raiz) si los grupos marcados como "no
    # cuadra" siguen de verdad sin resolver, o si el dinero real ya
    # lo cubrió por completo (caso real: DOC-19/DOC-05, donde el ajuste
    # manual del contable cubría exactamente el sobrante nominal --
    # avisar aquí habría sido un falso aviso, ya resuelto).
    pendiente_por_doc_temp = (calculado['importe_total_con_iva'] - calculado['total_cobrado_nuevo']).round(2)
    pendiente_por_doc_temp.index = calculado['documento']
    pendiente_dict_temp = pendiente_por_doc_temp.to_dict()
    for documentos_grupo, doc_raiz, _sobrante_nominal in avisos_sobrante_grupo:
        pendiente_total_grupo = round(sum(pendiente_dict_temp.get(d, 0) for d in documentos_grupo), 2)
        if abs(pendiente_total_grupo) > 0.01:
            print(f"⚠️ AVISO [{nombre_fac}]: grupo de compensación {documentos_grupo} no cuadra del todo "
                  f"contra su raíz {doc_raiz} -- sobran {pendiente_total_grupo:.2f}€, se mantienen pendientes")

    actual_bd = pd.read_sql(f"""
        SELECT ft.documento, ft.importe_total_con_iva AS importe_actual,
               ft.documento_original AS documento_original_actual,
               ft.tiene_recibo AS tiene_recibo_actual,
               ft.fecha_factura AS fecha_factura_actual,
               COALESCE(SUM(c.{col_importe_cobro}), 0) AS total_cobrado_actual
        FROM {tabla_facturas_totales} ft
        LEFT JOIN {tabla_cobros} c ON ft.documento = c.documento
        GROUP BY ft.documento, ft.importe_total_con_iva, ft.documento_original, ft.tiene_recibo, ft.fecha_factura
    """, engine)
    # PostgreSQL NUMERIC llega como Decimal (dtype object) — forzar a
    # float antes de cualquier operación numérica (.round(), comparación)
    actual_bd['importe_actual'] = pd.to_numeric(actual_bd['importe_actual'], errors='coerce')
    actual_bd['total_cobrado_actual'] = pd.to_numeric(actual_bd['total_cobrado_actual'], errors='coerce')

    comparacion = calculado.merge(actual_bd, on='documento', how='left')
    comparacion['importe_actual'] = comparacion['importe_actual'].round(2)
    comparacion['total_cobrado_actual'] = comparacion['total_cobrado_actual'].fillna(-1).round(2)  # -1 = no existía antes
    # Comparar documento_original con != directo NO es fiable cuando
    # hay None de por medio: en una columna de texto de pandas, "None
    # != None" puede dar True en vez de False (comportamiento real
    # comprobado, no un caso raro) -- se normalizan ambos lados a un
    # mismo texto fijo antes de comparar, mismo patrón ya usado hoy
    # con importes, fechas, id_inmueble y tipo_operacion.
    doc_original_nuevo_normalizado = comparacion['documento_original'].fillna('__NULL__')
    doc_original_actual_normalizado = comparacion['documento_original_actual'].fillna('__NULL__')

    # Fechas y reparto de los cobros (además del total cobrado)
    cobros_bd = pd.read_sql(f"SELECT documento, numero_cuota, {col_fecha_cobro} AS fecha, {col_importe_cobro} AS importe FROM {tabla_cobros}", engine)
    h_cobros_nueva = _huella_cobros(cobros_df.rename(columns={col_fecha_cobro: 'fecha', col_importe_cobro: 'importe'}))
    h_cobros_actual = _huella_cobros(cobros_bd)
    cobros_distintos = comparacion['documento'].map(h_cobros_nueva).fillna('') != comparacion['documento'].map(h_cobros_actual).fillna('')

    comparacion['ha_cambiado'] = (
        (comparacion['importe_total_con_iva'].round(2) != comparacion['importe_actual']) |
        (comparacion['total_cobrado_nuevo'] != comparacion['total_cobrado_actual']) |
        (doc_original_nuevo_normalizado != doc_original_actual_normalizado) |
        (comparacion['tiene_recibo'] != comparacion['tiene_recibo_actual']) |
        cobros_distintos
    )
    documentos_afectados = set(comparacion[comparacion['ha_cambiado']]['documento'])

    # Expandir: si un documento va a borrarse/reinsertarse, cualquier
    # otro documento con documento_original apuntando a él debe moverse
    # EN EL MISMO LOTE — si no, al borrar el primero, la referencia del
    # segundo queda colgando a mitad de transacción (violación de la FK
    # autorreferenciada), aunque el segundo no haya cambiado de valor.
    apuntan_a_afectados = set(
        facturas_totales_df[facturas_totales_df['documento_original'].isin(documentos_afectados)]['documento']
    )
    documentos_afectados |= apuntan_a_afectados

    # Facturas que están en la base de datos pero YA NO existen en el origen (se borraron en FacturaPlus):
    # se eliminan también de aquí, con sus cobros. Se compara con las CABECERAS del origen, no con las
    # líneas: una factura cuyas líneas se descartan (p. ej. sin artículo) sigue existiendo y no se toca.
    documentos_desaparecidos = set(actual_bd['documento']) - set(facturas_totales_df['documento'])
    if len(documentos_desaparecidos) > MAX_FACTURAS_A_BORRAR:
        raise RuntimeError(f"[{nombre_fac}] Faltan en el origen {len(documentos_desaparecidos)} facturas que SÍ están en la base de datos "
                           f"(límite de seguridad: {MAX_FACTURAS_A_BORRAR}). No se ha borrado nada. Si es correcto, sube MAX_FACTURAS_A_BORRAR en la Celda 1; "
                           f"si no, revisa que el DBF de origen esté completo. Primeras: {sorted(documentos_desaparecidos)[:5]}")
    documentos_afectados |= documentos_desaparecidos
    if documentos_desaparecidos:
        print(f"[{nombre_fac}] Facturas que ya no existen en el origen y se eliminan: {sorted(documentos_desaparecidos)}")

    # Fecha de la factura: si no estaba guardada o ha cambiado, se actualiza SOLA, sin tocar importes ni cobros.
    # Las facturas que ya se reescriben por otro motivo llevan la fecha nueva en su fila.
    fecha_nueva = pd.to_datetime(comparacion['fecha_factura'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('NULL')
    fecha_guardada = pd.to_datetime(comparacion['fecha_factura_actual'], errors='coerce').dt.strftime('%Y-%m-%d').fillna('NULL')
    solo_fecha = comparacion[(fecha_nueva != fecha_guardada) & (comparacion['total_cobrado_actual'] != -1)
                             & ~comparacion['documento'].isin(documentos_afectados)]

    print(f"[{nombre_fac}] Documentos con cambio real respecto a lo ya guardado: {len(documentos_afectados)} de {len(comparacion)}")

    # ─── Escribir solo lo afectado ───
    facturas_afectadas = facturas_totales_df[facturas_totales_df['documento'].isin(documentos_afectados)]
    cobros_afectados = cobros_df[cobros_df['documento'].isin(documentos_afectados)]

    if len(documentos_afectados) > 0:
        docs_lista = tuple(sorted(documentos_afectados))
        # Contar filas ANTIGUAS antes de borrarlas -- facturas_totales
        # ya tenemos su conteo en 'actual_bd' (total_cobrado_actual
        # distinto de -1 marca "ya existía"); cobros hay que contarlo
        # aparte, con una consulta directa, porque no lo teníamos en
        # memoria
        facturas_eliminadas = int((actual_bd['documento'].isin(docs_lista) & (actual_bd['total_cobrado_actual'] != -1)).sum())
        # Todo en UNA sola transacción: borrar cobros y facturas, e insertar facturas y cobros.
        # Si algo falla a mitad, no queda nada a medias (antes, un fallo entre el borrado y las
        # inserciones dejaba facturas sin cobros, es decir, \"Pendiente\" de forma errónea).
        with engine.begin() as conn:
            r = conn.execute(text(f"SELECT COUNT(*) FROM {tabla_cobros} WHERE documento IN :docs").bindparams(docs=docs_lista))
            cobros_eliminados = r.scalar()
            conn.execute(text(f"DELETE FROM {tabla_cobros} WHERE documento IN :docs").bindparams(docs=docs_lista))
            conn.execute(text(f"DELETE FROM {tabla_facturas_totales} WHERE documento IN :docs").bindparams(docs=docs_lista))
            facturas_afectadas.to_sql(tabla_facturas_totales, conn, if_exists='append', index=False)
            cobros_afectados.to_sql(tabla_cobros, conn, if_exists='append', index=False)
        print(f"[{nombre_fac}] Actualizados: {len(facturas_afectadas)} facturas, {len(cobros_afectados)} filas "
              f"(eliminadas antes: {facturas_eliminadas} facturas, {cobros_eliminados} filas de cobro)\n")
    else:
        print(f"[{nombre_fac}] Actualizados: 0 facturas, 0 filas\n")

    if len(solo_fecha) > 0:
        fechas_solas = pd.to_datetime(solo_fecha['fecha_factura'], errors='coerce')
        with engine.begin() as conn:
            conn.execute(text(f"UPDATE {tabla_facturas_totales} SET fecha_factura = :f WHERE documento = :d"),
                         [{'f': (None if pd.isna(f) else f.date()), 'd': d} for d, f in zip(solo_fecha['documento'], fechas_solas)])
        print(f"[{nombre_fac}] Fecha de factura guardada o corregida en {len(solo_fecha)} facturas (sin tocar importes ni cobros)\n")


## Celda 9 — Ejecutar para clientes


In [ ]:
# Los 3 primeros parámetros son los ficheros DBF de origen; los 2
# siguientes, las tablas donde se guarda el resultado; los 4 últimos,
# los nombres de columna que cambian entre clientes y proveedores
# (código de tercero, columnas del cobro y fecha de la factura en la cabecera)
procesar_incremental(
    "Facclit.dbf", "Recibos.dbf", "Pagdev.dbf",
    "facturas_totales_clientes", "cobros_clientes",
    "CCODCLI", "fecha_cobro", "importe_cobro", "DFECFAC"
)


## Celda 10 — Ejecutar para proveedores


In [ ]:
# Misma función que para clientes, con los ficheros y nombres de
# columna correspondientes al lado de proveedores
procesar_incremental(
    "Facprot.dbf", "Recibosp.dbf", "Pagdevp.dbf",
    "facturas_totales_proveedores", "pagos_proveedores",
    "CCODPRO", "fecha_pago", "importe_pago", "DSUFECFAC"
)


## Celda 11 — Verificación general

Consulta las vistas `estado_cobro_clientes`/`estado_pago_proveedores`
agrupando por estado. **La comprobación más importante**: el total
pendiente de "Cobrada"/"Pagada"/"Saldada" debe dar siempre **0,00€
exacto** — es la garantía de que ninguna factura ha quedado cobrada o
pagada de más en ningún punto del proceso.


In [ ]:
# Agrupar por estado y sumar 'pendiente' -- la comprobación clave es
# que Cobrada/Pagada/Saldada den siempre 0,00€ exacto
# Dos decimales fijos, sin notación científica (el ruido de coma flotante tipo -1.9e-13 es 0,00)
pd.options.display.float_format = '{:,.2f}'.format

verificacion_clientes = pd.read_sql("""
    SELECT estado, COUNT(*) AS n_facturas, ROUND(SUM(pendiente)::numeric, 2) AS total_pendiente
    FROM estado_cobro_clientes GROUP BY estado ORDER BY n_facturas DESC
""", engine)
print("=== Clientes ===")
print(verificacion_clientes.to_string(index=False))

verificacion_proveedores = pd.read_sql("""
    SELECT estado, COUNT(*) AS n_facturas, ROUND(SUM(pendiente)::numeric, 2) AS total_pendiente
    FROM estado_pago_proveedores GROUP BY estado ORDER BY n_facturas DESC
""", engine)
print("\n=== Proveedores ===")
print(verificacion_proveedores.to_string(index=False))


## Celda 12 — Cuadre final: ¿está todo el dinero contado y explicado?

Cada línea de `Facclil.dbf` (clientes) y `Facprol.dbf` (proveedores) acaba
en **un solo sitio**:

| Grupo | Qué es |
|---|---|
| **A** | Imputada a un inmueble |
| **B** | Sin inmueble, **por diseño**: compraventas, obra, impuestos de la compra, estructura, otros |
| **C** | Sin inmueble, **pendiente**: deberá asociarse a algún inmueble (repartir, dar de alta un solar...) |
| **D** | Sin inmueble, de **inmuebles antiguos** que ya no están en cartera |
| **E** | Sin inmueble y **sin clasificar**: hay que decidir qué es |

A + B + C + D + E cuadra al céntimo con el DBF (menos lo que descarta el
cargador) y con la base de datos. Clientes y proveedores se cuadran por
separado, porque ingresos y gastos no se suman entre sí.

**El informe tiene cinco secciones:** 1 Veredicto (¿cuadra? ¿hay algo que
hacer?) · 2 Resumen (adónde ha ido cada euro) · 3 Comprobaciones ·
4 Lo que requiere tu atención (sin clasificar, avisos y grupo C) ·
5 Anexo (las combinaciones que explican B y D, para consultar).

**Red de seguridad:** un artículo sin ningún inmueble actual se daría por
«antiguo» sin más. Para que no se escondan cosas de una promoción que sí
tienes, si su nombre menciona alguna palabra de `PROMOCIONES_ACTUALES`
(el nombre de una promoción en curso, por ejemplo) sale como **E · sin clasificar** en lugar de D,
y el veredicto te avisa.

**Lo único que se mantiene a mano**, al principio de la celda:
`PENDIENTES_DE_REPARTIR` (una línea por combinación que queda sin inmueble
pero deberá repartirse) y `MOTIVO_POR_ARTICULO` (un motivo explícito para
los artículos que el tipo de operación no explica por sí solo).
`PROMOCIONES_ACTUALES` alimenta la red de seguridad. `ANEXO_COMBINACIONES` y
`MOSTRAR_ANEXO` ajustan cuánto se enseña en la sección 5, donde lo decidido por
artículo se enseña por artículo y el resto combinación por combinación.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CUADRE FINAL · ¿está todo el dinero contado y explicado?
#
# Cada línea de factura (Facclil.dbf y Facprol.dbf) acaba en un solo sitio:
#   A · imputada a un inmueble
#   B · sin inmueble, por diseño (compraventas, obra, estructura...)
#   C · sin inmueble, PENDIENTE: deberá asociarse a algún inmueble
#   D · sin inmueble, de inmuebles ANTIGUOS que ya no están en cartera
#   E · sin inmueble y SIN CLASIFICAR: hay que decidir qué es
# La suma A + B + C + D + E cuadra al céntimo con el DBF y con la base de datos.
# ═══════════════════════════════════════════════════════════════════════

# ── AJUSTES DEL INFORME ─────────────────────────────────────────────────
ANEXO_COMBINACIONES = 8      # cuántas combinaciones enseñar de los motivos grandes en el anexo
MOSTRAR_ANEXO = True         # False: oculta la sección 5 (el anexo)

# ── LO ÚNICO QUE SE MANTIENE A MANO (1/2): combinaciones pendientes de repartir ──
# Combinaciones que quedan SIN inmueble pero DEBEN repartirse algún día.
# Clave: (cref, cprop) -- cprop '' son las líneas sin etiqueta.
# Valor: el grupo de unidades entre las que se repartiría.
PENDIENTES_DE_REPARTIR = {
    # Ejemplos (sustitúyelos por tus combinaciones reales; con el diccionario vacío no hay nada pendiente):
    # ('0001', 'ETIQUETA 1'): 'Promoción 1 - viviendas',
    # ('0002', ''):           'Promoción 1 - zonas comunes',   # cprop '' = líneas sin etiqueta
    # ('0003', 'BLOQUE A'):   'Promoción 2 - bloque A',
    # ('0004', ''):           'Promoción 2 - complejo entero',
}

# RED DE SEGURIDAD: palabras del nombre de un artículo que delatan una promoción que SÍ sigue en cartera.
# Un artículo sin ningún inmueble actual se daba por «antiguo» sin más. Ahora, si su nombre menciona una
# de estas palabras, sale como SIN CLASIFICAR para que decidas qué es (añádelo a PENDIENTES_DE_REPARTIR
# si deberá repartirse, o a MOTIVO_POR_ARTICULO si de verdad es antiguo). Amplía la lista cuando haga falta.
PROMOCIONES_ACTUALES = ('PROMOCION 1', 'PROMOCION 2')      # ejemplo: pon aquí palabras que delaten tus promociones en cartera

# Tipos de operación operativos: lo no imputado de estos tipos se explica
# por inmuebles antiguos, o queda como SIN CLASIFICAR (hay que decidir).
TIPOS_OPERATIVOS = ['ALQUILER', 'GASTOS_EXPLOTACION', 'GASTOS_MANTENIMIENTO', 'IMPUESTOS_TASAS']
# Una combinación operativa sin movimientos desde esta fecha se considera de un inmueble antiguo
ULTIMO_MOVIMIENTO_DESDE = pd.Timestamp('2022-01-01')
PATRON_UNIDAD = re.compile(r'^(PUERTA|PTA|PLAZA|PLZ|PZA|PG|TRASTERO|TR|VIV|OFICINA|OFI)\s*-?\s*\d+\s*[A-Z]?$')

# ── Motivos (la etiqueta es lo que se ve en el informe) ─────────────────
# C · pendiente de asociar a algún inmueble
M_PENDIENTE = 'Pendiente de repartir entre las unidades de un grupo'
M_SOLAR_PENDIENTE = 'Solar actual pendiente de dar de alta (su IBI irá a él)'
M_VARIOS_SIN_GRUPO = 'Varios inmuebles de distintas promociones (sin grupo de reparto)'
# B · por diseño (el coste de adquisición de cada inmueble se carga a mano en la tabla inmuebles)
M_COMPRAVENTA = 'Compraventa de inmuebles (coste de adquisición a mano)'
M_PROMOCION = 'Gastos de promoción y obra (coste de adquisición a mano)'
M_IMPUESTOS_COMPRA = 'Impuestos de la compra (ITP, AJD): ya en el coste'
M_ESTRUCTURA = 'Gastos de estructura (no son de ningún inmueble)'
M_OTROS = 'Otros movimientos no operativos (reformas de terceros, fianzas)'
# D · inmuebles antiguos
M_ANT_PLUSVALIAS = 'Plusvalías de inmuebles vendidos'
M_ANT_PROMOCIONES = 'Promociones, viviendas y solares ya vendidos'
M_ANT_UNIDAD = 'Unidad concreta que ya no está en cartera'
M_ANT_SIN_MOV = f'Sin movimientos desde {ULTIMO_MOVIMIENTO_DESDE:%Y} (comunes antiguos...)'
M_ANT_ARTICULO = 'Resto: artículo sin ningún inmueble actual'
# E · sin clasificar
M_SIN_CLASIFICAR = 'SIN CLASIFICAR: hay que decidir'
M_MENCIONA_ACTUAL = 'SIN CLASIFICAR: parece antiguo pero menciona una promoción actual'
M_SIN_TIPO = 'SIN CLASIFICAR: artículo sin tipo de operación'
M_IMPUTADA = 'IMPUTADA A UN INMUEBLE'

GRUPOS = {
    'A': 'A · IMPUTADO a un inmueble',
    'B': 'B · NO imputado, por diseño: no es de ningún inmueble de la cartera',
    'C': 'C · NO imputado, PENDIENTE: deberá asociarse a algún inmueble',
    'D': 'D · NO imputado, de inmuebles ANTIGUOS: ya no están en cartera',
    'E': 'E · NO imputado, SIN CLASIFICAR: hay que decidir qué es',
}
GRUPO_DE_MOTIVO = {M_COMPRAVENTA: 'B', M_PROMOCION: 'B', M_IMPUESTOS_COMPRA: 'B', M_ESTRUCTURA: 'B', M_OTROS: 'B',
                   M_PENDIENTE: 'C', M_SOLAR_PENDIENTE: 'C', M_VARIOS_SIN_GRUPO: 'C',
                   M_ANT_PLUSVALIAS: 'D', M_ANT_PROMOCIONES: 'D', M_ANT_UNIDAD: 'D', M_ANT_SIN_MOV: 'D', M_ANT_ARTICULO: 'D',
                   M_SIN_CLASIFICAR: 'E', M_MENCIONA_ACTUAL: 'E', M_SIN_TIPO: 'E'}
ORDEN_MOTIVOS = list(GRUPO_DE_MOTIVO)

# Tipos que, sin inmueble, ya se explican por sí mismos
MOTIVO_POR_TIPO = {'COMPRAVENTA': M_COMPRAVENTA, 'GASTOS_PROMOCION': M_PROMOCION, 'GASTOS_ESTRUCTURA': M_ESTRUCTURA, 'OTROS': M_OTROS}

# ── LO ÚNICO QUE SE MANTIENE A MANO (2/2): motivo explícito por artículo ──
# Para lo que no encaja por tipo de operación. Tiene prioridad sobre las reglas
# automáticas. La explicación de cada uno es el nombre del propio artículo.
MOTIVO_POR_ARTICULO = {
    # Ejemplos (sustitúyelos por tus artículos reales):
    # '0010': M_OTROS,               # operación poco clara
    # '0011': M_SOLAR_PENDIENTE,     # IBI de un solar actual que falta dar de alta
    # '0012': M_VARIOS_SIN_GRUPO,    # gasto que afecta a inmuebles de distintas promociones
    # '0013': M_ANT_PLUSVALIAS,      # plusvalías de inmuebles ya vendidos
    # '0014': M_IMPUESTOS_COMPRA,    # impuestos de la compra: ya en el coste
    # '0015': M_ANT_PROMOCIONES,     # gastos de una promoción ya vendida
}

# ═══════════════════════════════════════════════════════════════════════
# PRESENTACIÓN (solo da formato; no calcula nada)
# ═══════════════════════════════════════════════════════════════════════
ANCHO, COL = 116, 76
ABREV_TIPO = {'ALQUILER': 'alquiler', 'GASTOS_EXPLOTACION': 'explotación', 'GASTOS_MANTENIMIENTO': 'mantenim.', 'IMPUESTOS_TASAS': 'impuestos',
              'COMPRAVENTA': 'compraventa', 'GASTOS_PROMOCION': 'promoción', 'GASTOS_ESTRUCTURA': 'estructura', 'OTROS': 'otros', '(SIN TIPO)': '(sin tipo)'}


def _es(x, dec=2):
    """1234567.8 -> '1.234.567,80' (formato español)."""
    return f"{x:,.{dec}f}".replace(',', '§').replace('.', ',').replace('§', '.')


def fe(cent):
    return _es(cent / 100)


def fnum(n):
    return f"{int(n):,}".replace(',', '.')


def _pl(n, singular, plural):
    return f"{fnum(n)} {singular if n == 1 else plural}"


def pct(parte, total):
    return f"{_es(parte / total * 100, 1):>6} %" if total else ''


def seccion(texto):
    print(f"\n{'═' * ANCHO}\n {texto}\n{'═' * ANCHO}")


def subseccion(texto):
    print(f"\n── {texto} " + '─' * max(2, ANCHO - len(texto) - 4))


def fila(texto, comb=None, n=None, cent=None, p='', sangria=0):
    t = f"{' ' * sangria}{texto}"[:COL].ljust(COL)
    c = f"{fnum(comb):>5}" if comb is not None else ' ' * 5
    l = f"{fnum(n):>8}" if n is not None else ' ' * 8
    i = f"{fe(cent):>16}" if cent is not None else ' ' * 16
    print(f"{t} {c} {l} {i} {p}")


def cabecera_resumen():
    print(f"{'':<{COL}} {'comb.':>5} {'líneas':>8} {'importe (€)':>16} {'%':>8}")


def cabecera_comb(sangria=3):
    print(f"{' ' * sangria}{'artículo':<5} {'etiqueta':<14} {'tipo':<11} {'descripción':<46} {'líneas':>6} {'importe (€)':>17}")


def fila_comb(r, sangria=3):
    etiqueta = r.cprop if r.cprop else '(sin etiqueta)'
    print(f"{' ' * sangria}{r.CREF:<5} {etiqueta[:14]:<14} {ABREV_TIPO.get(r.tipo, r.tipo):<11} {str(r.descripcion)[:46]:<46} {fnum(r.lineas):>6} {fe(r.cent):>15} €")


def combinaciones(df):
    """Una fila por artículo + etiqueta + tipo, con líneas e importe, de mayor a menor importe."""
    d = df.groupby(['CREF', 'cprop', 'tipo']).agg(lineas=('cent', 'size'), cent=('cent', 'sum')).reset_index()
    d['descripcion'] = d['CREF'].map(descripcion_articulo).fillna('')
    return d.reindex(d['cent'].abs().sort_values(ascending=False).index)


def por_articulo(df):
    """Una fila por artículo (suma todas sus etiquetas), de mayor a menor importe."""
    d = df.groupby('CREF').agg(combinaciones=('cprop', 'nunique'), lineas=('cent', 'size'), cent=('cent', 'sum')).reset_index()
    d['descripcion'] = d['CREF'].map(descripcion_articulo).fillna('')
    return d.reindex(d['cent'].abs().sort_values(ascending=False).index)


def cabecera_art(sangria=3):
    print(f"{' ' * sangria}{'artículo':<8} {'combinaciones':>13} {'descripción':<46} {'líneas':>6} {'importe (€)':>17}")


def fila_art(r, sangria=3):
    print(f"{' ' * sangria}{r.CREF:<8} {fnum(r.combinaciones):>13} {str(r.descripcion)[:46]:<46} {fnum(r.lineas):>6} {fe(r.cent):>15} €")


def imprimir_veredicto(rc, rp):
    ok = rc['ok_total'] and rp['ok_total']
    avisos = rc['avisos'] + rp['avisos']
    sin_cli, sin_pro = rc['sin_clasificar'], rp['sin_clasificar']
    grupo_c = lambda r: int(r['por_motivo'].loc[r['por_motivo']['motivo'].map(GRUPO_DE_MOTIVO) == 'C', 'cent'].sum())

    seccion("CUADRE FINAL · ¿está todo el dinero contado y explicado?")
    print(" Secciones:  1 Veredicto · 2 Resumen · 3 Comprobaciones · 4 Lo que requiere tu atención · 5 Anexo")
    print("\n 1 · VEREDICTO")
    if ok:
        print("   ✅ El dinero cuadra: lo imputado + lo no imputado = lo facturado en el DBF = lo que hay en la base de datos")
    else:
        print("   ❌ El dinero NO cuadra con el DBF o con la base de datos   → ver la sección 3")
    if avisos == 0:
        print("   ✅ Ninguna combinación pendiente de repartir va todavía a un inmueble")
    else:
        print(f"   ⚠️  {_pl(avisos, 'combinación pendiente de repartir aún va', 'combinaciones pendientes de repartir aún van')} a un inmueble   → ver la sección 4")
    if sin_cli + sin_pro == 0:
        print("   ✅ Todo lo no imputado tiene un motivo (sin clasificar: 0,00 €)")
    else:
        print(f"   ❗ Sin clasificar: clientes {fe(sin_cli)} € · proveedores {fe(sin_pro)} €   → ver la sección 4")
    print(f"   ℹ️  Pendiente de asociar a algún inmueble (grupo C): clientes {fe(grupo_c(rc))} € · proveedores {fe(grupo_c(rp))} €")
    print("\n   Cómo leer: cada línea de factura tiene un artículo y, a veces, una etiqueta de unidad (PUERTA43, PG1...).")
    print("   Artículo + etiqueta = una «combinación». Cada combinación acaba en A (con inmueble) o en B, C, D o E (sin inmueble, con su motivo).")


def imprimir_resumen(r):
    subseccion(r['titulo'])
    cabecera_resumen()
    fila(f"Total en el DBF ({r['fichero']})", None, r['origen_n'], r['origen_cent'])
    hay = False
    for texto, n, x in r['descartes']:
        if n:
            hay = True
            fila(f"− Descartado por el cargador: {texto.replace('CREF', 'artículo')}", None, n, -x, sangria=2)
    if not hay:
        fila("− Descartado por el cargador: nada", None, 0, 0, sangria=2)
    fila("= Entra en el proceso", None, r['entra_n'], r['entra_c'], pct(r['entra_c'], r['entra_c']))
    print()
    total, por = r['entra_c'], r['por_motivo']
    fila(GRUPOS['A'], None, r['imp_n'], r['imp_c'], pct(r['imp_c'], total))
    for g in 'BCDE':
        sub = por[por['motivo'].map(lambda m: GRUPO_DE_MOTIVO.get(m, 'E')) == g]
        cg = int(sub['cent'].sum())
        fila(GRUPOS[g], int(sub['combinaciones'].sum()), int(sub['lineas'].sum()), cg, pct(cg, total))
        for m in sub.itertuples():
            fila(m.motivo, m.combinaciones, m.lineas, m.cent, pct(m.cent, total), sangria=5)
    fila("TOTAL  (A + B + C + D + E)", None, r['entra_n'], r['imp_c'] + int(por['cent'].sum()), pct(r['imp_c'] + int(por['cent'].sum()), total))


def pendiente_por_grupo(r):
    """Líneas e importe de lo pendiente de repartir, por grupo de reparto (los nombres de PENDIENTES_DE_REPARTIR)."""
    lin = r['lin']
    pend = lin[(~lin['imputada']) & (lin['motivo'] == M_PENDIENTE)]
    if not len(pend):
        return {}
    pend = pend.assign(grupo=[PENDIENTES_DE_REPARTIR[(a, b)] for a, b in zip(pend['CREF'], pend['cprop'])])
    return {g: (len(x), int(x['cent'].sum())) for g, x in pend.groupby('grupo')}


def imprimir_pendiente_por_promocion(rc, rp):
    """Resumen de lo pendiente de repartir: el total de cada promoción y, debajo, su desglose por subdivisión.
    El nombre de cada grupo de reparto se lee como «Promoción - subdivisión» (por ejemplo, «Promoción 1 - garajes»)."""
    subseccion("PENDIENTE DE REPARTIR · por promoción y subdivisión (clientes y proveedores)")
    datos = {'c': pendiente_por_grupo(rc), 'p': pendiente_por_grupo(rp)}
    if not datos['c'] and not datos['p']:
        print("✅ Nada pendiente de repartir.")
        return
    arbol = {}                                     # promoción → subdivisión → {'c': (líneas, cent), 'p': (líneas, cent)}
    for lado, grupos in datos.items():
        for nombre, valor in grupos.items():
            promo, _, sub = nombre.partition(' - ')
            arbol.setdefault(promo.strip(), {}).setdefault(sub.strip(), {'c': (0, 0), 'p': (0, 0)})[lado] = valor
    ancho = 46
    suma = lambda a, b: (a[0] + b[0], a[1] + b[1])
    celda = lambda v: f"{fnum(v[0]) if v[0] else '':>7} {fe(v[1]) if v[0] else '':>16}"

    def linea(texto, c, p, sangria=0):
        print(f"{(' ' * sangria + texto)[:ancho].ljust(ancho)}  {celda(c)}   {celda(p)}")

    print(f"{'':<{ancho}}  {'CLIENTES':^24}   {'PROVEEDORES':^24}")
    print(f"{'':<{ancho}}  {'líneas':>7} {'importe (€)':>16}   {'líneas':>7} {'importe (€)':>16}")
    total_c = total_p = (0, 0)
    for promo in sorted(arbol):
        subs = arbol[promo]
        pc = pp = (0, 0)
        for s in subs.values():
            pc, pp = suma(pc, s['c']), suma(pp, s['p'])
        linea(promo, pc, pp)
        if any(subs):                              # la promoción tiene subdivisiones con nombre: se detallan
            for sub in sorted(subs):
                linea(sub or '(sin subdivisión)', subs[sub]['c'], subs[sub]['p'], sangria=4)
        total_c, total_p = suma(total_c, pc), suma(total_p, pp)
    print('─' * (ancho + 2 + 24 + 3 + 24))
    linea("TOTAL pendiente de repartir", total_c, total_p)
    resto = []
    for r, total in ((rc, total_c), (rp, total_p)):
        por = r['por_motivo']
        grupo_c = int(por.loc[por['motivo'].map(lambda m: GRUPO_DE_MOTIVO.get(m, 'E')) == 'C', 'cent'].sum())
        resto.append(grupo_c - total[1])
    if resto[0] or resto[1]:
        print(f"\n   Además, dentro del grupo C pero sin ser repartir (solares sin dar de alta, varios sin grupo): "
              f"clientes {fe(resto[0])} € · proveedores {fe(resto[1])} €")


def imprimir_comprobaciones(rc, rp):
    print(f"\n 3 · COMPROBACIONES (la diferencia debe ser 0)")
    print(f"   {'':<76} {'Clientes':>10} {'Proveedores':>13}")
    for a, b in zip(rc['checks'], rp['checks']):
        print(f"   {a['texto']:<76} {'✅' if a['ok'] else '❌':>9} {'✅' if b['ok'] else '❌':>12}")
    for nombre, r in (('Clientes', rc), ('Proveedores', rp)):
        for k in r['checks']:
            if not k['ok']:
                print(f"   ❌ {nombre} · {k['texto']}: {fe(k['dif_c'])} € | {fnum(k['dif_n'])} líneas")


def imprimir_atencion(r, nombre):
    lin = r['lin']
    no = lin[~lin['imputada']]
    subseccion(nombre)
    # E · sin clasificar
    sin = no[no['motivo'].map(lambda m: GRUPO_DE_MOTIVO.get(m, 'E')) == 'E']
    if len(sin):
        print(f"❗ SIN CLASIFICAR: {fe(int(sin['cent'].sum()))} € en {_pl(len(combinaciones(sin)), 'combinación', 'combinaciones')}. Hay que decidir qué son:")
        for m in (M_MENCIONA_ACTUAL, M_SIN_CLASIFICAR, M_SIN_TIPO):
            s2 = sin[sin['motivo'] == m]
            if len(s2):
                print(f"\n  ▸ {m} · {fe(int(s2['cent'].sum()))} €")
                cabecera_comb(4)
                for x in combinaciones(s2).itertuples():
                    fila_comb(x, 4)
        print("\n  Cómo resolverlo: añade la combinación a PENDIENTES_DE_REPARTIR (si deberá repartirse) o el artículo a MOTIVO_POR_ARTICULO (si es antiguo o por diseño).")
    else:
        print("✅ Nada sin clasificar.")
    # avisos
    aun = r['aun']
    if len(aun):
        print("\n⚠️  Estas combinaciones están en PENDIENTES_DE_REPARTIR pero TODAVÍA van a un inmueble:")
        for x in aun.groupby(['CREF', 'cprop', 'id_inmueble']).agg(lineas=('cent', 'size'), cent=('cent', 'sum')).reset_index().itertuples():
            print(f"   {x.CREF:<5} {(x.cprop or '(sin etiqueta)'):<14} → {x.id_inmueble}   {fnum(x.lineas):>5} líneas {fe(x.cent):>15} €")
        print("   Revisa la tabla puente (¿hay una fila general del artículo con inmueble en la base de datos?).")
    else:
        print("✅ Ninguna combinación pendiente de repartir va todavía a un inmueble.")
    # C · pendiente
    pend = no[no['motivo'].map(lambda m: GRUPO_DE_MOTIVO.get(m, 'E')) == 'C']
    if not len(pend):
        print("\n✅ Nada pendiente de asociar a algún inmueble.")
        return
    print(f"\nC · PENDIENTE de asociar a algún inmueble: {fe(int(pend['cent'].sum()))} € en {_pl(len(pend[['CREF', 'cprop']].drop_duplicates()), 'combinación', 'combinaciones')}")
    for m in (M_PENDIENTE, M_SOLAR_PENDIENTE, M_VARIOS_SIN_GRUPO):
        sub = pend[pend['motivo'] == m]
        if not len(sub):
            continue
        print(f"\n  ▸ {m} · {fe(int(sub['cent'].sum()))} €")
        if m == M_PENDIENTE:
            cabecera_comb(6)
            sub = sub.assign(grupo=[PENDIENTES_DE_REPARTIR[(a, b)] for a, b in zip(sub['CREF'], sub['cprop'])])
            for grupo in sorted(sub['grupo'].unique()):
                g = sub[sub['grupo'] == grupo]
                print(f"\n    {grupo}  ·  {_pl(len(g[['CREF', 'cprop']].drop_duplicates()), 'combinación', 'combinaciones')}  ·  {_pl(len(g), 'línea', 'líneas')}  ·  {fe(int(g['cent'].sum()))} €")
                for x in combinaciones(g).itertuples():
                    fila_comb(x, 6)
        else:
            print()
            cabecera_comb(4)
            for x in combinaciones(sub).itertuples():
                fila_comb(x, 4)


def imprimir_anexo(r, nombre):
    no = r['lin'][~r['lin']['imputada']]
    subseccion(nombre)
    # Lo que has decidido por artículo (plusvalías, promociones vendidas...) se enseña por artículo;
    # el resto, combinación por combinación (de mayor a menor importe).
    por_art = (M_ANT_PLUSVALIAS, M_ANT_PROMOCIONES)
    bloques = ((M_OTROS, None), (M_IMPUESTOS_COMPRA, None), (M_ESTRUCTURA, ANEXO_COMBINACIONES),
               (M_ANT_PLUSVALIAS, None), (M_ANT_PROMOCIONES, None), (M_ANT_UNIDAD, ANEXO_COMBINACIONES),
               (M_ANT_SIN_MOV, ANEXO_COMBINACIONES), (M_ANT_ARTICULO, ANEXO_COMBINACIONES))
    for m, limite in bloques:
        sub = no[no['motivo'] == m]
        if not len(sub):
            continue
        quien = f"{GRUPO_DE_MOTIVO[m]} · {m}"
        total = f"total {fe(int(sub['cent'].sum()))} €"
        if m in por_art:
            a = por_articulo(sub)
            n_comb = len(sub[['CREF', 'cprop']].drop_duplicates())
            print(f"\n▸ {quien} — {_pl(len(a), 'artículo', 'artículos')} ({_pl(n_comb, 'combinación', 'combinaciones')}) · {total}")
            cabecera_art()
            for x in a.itertuples():
                fila_art(x)
            continue
        d = combinaciones(sub)
        if limite is None or len(d) <= limite:
            cuantas = 'la única combinación' if len(d) == 1 else f"las {len(d)} combinaciones"
        else:
            cuantas = f"las {limite} de mayor importe, de {len(d)}"
        print(f"\n▸ {quien} — {cuantas} · {total}")
        cabecera_comb()
        for x in (d if limite is None else d.head(limite)).itertuples():
            fila_comb(x)


# ═══════════════════════════════════════════════════════════════════════
# CÁLCULO
# ═══════════════════════════════════════════════════════════════════════
articulos_bd = pd.read_sql("SELECT cref, cprop, id_inmueble FROM articulos_inmuebles", engine)
articulos_con_inmueble = set(articulos_bd.loc[articulos_bd['id_inmueble'].notna(), 'cref'])
descripcion_articulo = leer_dbf("Articulo.dbf")[['CREF', 'CDETALLE']].drop_duplicates('CREF').set_index('CREF')['CDETALLE']
todas = pd.concat([v['lineas'][['CREF', 'CPROP1', 'fecha']] for v in CUADRE.values()])
todas = todas.assign(CPROP1=todas['CPROP1'].fillna(''), fecha=pd.to_datetime(todas['fecha'], errors='coerce'))
ULTIMA = todas.groupby(['CREF', 'CPROP1'])['fecha'].max().to_dict()


def motivo_de(cref, cprop, tipo):
    if (cref, cprop) in PENDIENTES_DE_REPARTIR:
        return M_PENDIENTE
    if cref in MOTIVO_POR_ARTICULO:
        return MOTIVO_POR_ARTICULO[cref]
    if tipo in MOTIVO_POR_TIPO:
        return MOTIVO_POR_TIPO[tipo]
    if tipo not in TIPOS_OPERATIVOS:
        return M_SIN_TIPO
    if cref not in articulos_con_inmueble:
        texto = str(descripcion_articulo.get(cref, '')).upper()
        return M_MENCIONA_ACTUAL if any(p in texto for p in PROMOCIONES_ACTUALES) else M_ANT_ARTICULO
    ultima = ULTIMA.get((cref, cprop))
    if pd.notna(ultima) and ultima < ULTIMO_MOVIMIENTO_DESDE:
        return M_ANT_SIN_MOV
    if PATRON_UNIDAD.match(cprop.upper().strip()):
        return M_ANT_UNIDAD
    return M_SIN_CLASIFICAR


def calcular(nombre, titulo, tabla_bd):
    c = CUADRE[nombre]
    lin = c['lineas'].copy()
    lin['cent'] = a_centimos(lin['NTOTLINEA'])
    lin['cprop'] = lin['CPROP1'].fillna('')
    lin['tipo'] = lin['tipo_operacion'].fillna('(SIN TIPO)')
    lin['imputada'] = lin['id_inmueble'].notna()
    combos = lin[~lin['imputada']].groupby(['CREF', 'cprop', 'tipo']).size().reset_index(name='n')
    combos['motivo'] = [motivo_de(a, b, t) for a, b, t in zip(combos['CREF'], combos['cprop'], combos['tipo'])]
    lin = lin.merge(combos[['CREF', 'cprop', 'tipo', 'motivo']], on=['CREF', 'cprop', 'tipo'], how='left')
    lin['motivo'] = lin['motivo'].where(~lin['imputada'], M_IMPUTADA)

    desc_n = sum(n for _, n, _ in c['descartes'])
    desc_c = sum(x for _, _, x in c['descartes'])
    entra_n, entra_c = len(lin), int(lin['cent'].sum())
    imp_n, imp_c = int(lin['imputada'].sum()), int(lin.loc[lin['imputada'], 'cent'].sum())
    no_n, no_c = entra_n - imp_n, entra_c - imp_c
    por_motivo = (lin[~lin['imputada']].assign(comb=lambda d: d['CREF'] + '|' + d['cprop'])
                  .groupby('motivo').agg(lineas=('cent', 'size'), cent=('cent', 'sum'), combinaciones=('comb', 'nunique'))
                  .reset_index())
    por_motivo['orden'] = por_motivo['motivo'].map({m: i for i, m in enumerate(ORDEN_MOTIVOS)}).fillna(len(ORDEN_MOTIVOS))
    por_motivo = por_motivo.sort_values('orden').drop(columns='orden').reset_index(drop=True)

    bd = pd.read_sql(f"SELECT id_inmueble, importe_linea FROM {tabla_bd}", engine)
    bd_c = a_centimos(bd['importe_linea'])
    bd_n, bd_total, bd_imp = len(bd), int(bd_c.sum()), int(bd_c[bd['id_inmueble'].notna().values].sum())
    suma_motivos = imp_c + int(por_motivo['cent'].sum())
    pruebas = [
        ("La suma de los motivos es igual a lo que entra", suma_motivos - entra_c, 0),
        ("Ningún cruce ha duplicado ni perdido líneas (entra = DBF − descartado)", entra_c - (c['origen_cent'] - desc_c), entra_n - (c['origen_n'] - desc_n)),
        ("Base de datos = notebook · total", bd_total - entra_c, bd_n - entra_n),
        ("Base de datos = notebook · imputado a un inmueble", bd_imp - imp_c, 0),
        ("Base de datos = notebook · sin inmueble", (bd_total - bd_imp) - no_c, 0),
    ]
    checks = [{'texto': t, 'ok': dc == 0 and dn == 0, 'dif_c': dc, 'dif_n': dn} for t, dc, dn in pruebas]

    # Combinaciones que se esperan SIN inmueble pero aún tienen líneas imputadas
    lin['es_pendiente'] = [(a, b) in PENDIENTES_DE_REPARTIR for a, b in zip(lin['CREF'], lin['cprop'])]
    aun = lin[lin['es_pendiente'] & lin['imputada']]
    avisos = aun[['CREF', 'cprop']].drop_duplicates().shape[0]
    sin_clasificar = int(por_motivo.loc[por_motivo['motivo'].map(lambda m: GRUPO_DE_MOTIVO.get(m, 'E')) == 'E', 'cent'].sum())
    return {'titulo': titulo, 'fichero': c['fichero'], 'origen_n': c['origen_n'], 'origen_cent': c['origen_cent'], 'descartes': c['descartes'],
            'entra_n': entra_n, 'entra_c': entra_c, 'imp_n': imp_n, 'imp_c': imp_c,
            'ok_total': all(k['ok'] for k in checks), 'checks': checks, 'avisos': avisos, 'aun': aun,
            'sin_clasificar': sin_clasificar, 'lin': lin, 'por_motivo': por_motivo}


res_cli = calcular('clientes', "CLIENTES · todo lo facturado", 'facturacion_clientes')
res_pro = calcular('proveedores', "PROVEEDORES · todo lo gastado", 'facturacion_proveedores')

# ═══════════════════════════════════════════════════════════════════════
# INFORME
# ═══════════════════════════════════════════════════════════════════════
imprimir_veredicto(res_cli, res_pro)
seccion("2 · RESUMEN: adónde ha ido cada euro (clientes y proveedores se cuadran por separado)")
imprimir_resumen(res_cli)
imprimir_resumen(res_pro)
imprimir_pendiente_por_promocion(res_cli, res_pro)
imprimir_comprobaciones(res_cli, res_pro)
seccion("4 · LO QUE REQUIERE TU ATENCIÓN")
imprimir_atencion(res_cli, "CLIENTES")
imprimir_atencion(res_pro, "PROVEEDORES")
if MOSTRAR_ANEXO:
    seccion(f"5 · ANEXO: las combinaciones que explican B y D, para consultar (ajusta ANEXO_COMBINACIONES = {ANEXO_COMBINACIONES})")
    print(" Compraventa y promoción (grupo B) no se listan: son cientos de combinaciones sueltas.")
    imprimir_anexo(res_cli, "CLIENTES")
    imprimir_anexo(res_pro, "PROVEEDORES")

total_ok = res_cli['ok_total'] and res_pro['ok_total']
avisos = res_cli['avisos'] + res_pro['avisos']
sin = res_cli['sin_clasificar'] + res_pro['sin_clasificar']
print(f"\n{'═' * ANCHO}\n RESULTADO:  {'✅' if total_ok else '❌'} cuadra   ·   {'✅' if avisos == 0 else '⚠️ '} avisos: {avisos}   ·   "
      f"{'✅' if sin == 0 else '❗'} sin clasificar: {fe(sin)} €\n{'═' * ANCHO}")
